In [1]:
# 1. Install necessary libraries
!pip install -q datasets polars kagglehub scikit-learn

import os
import glob
import polars as pl
import kagglehub
from datasets import load_dataset

print("Environment setup ready!")

Environment setup ready!


In [2]:
print("="*60)
print("1. LOADING MAIN TRAINING DATASET (Hugging Face: MeAJOR)")
print("="*60)
# Main dataset used for train/val/test splits
ds_main = load_dataset("zefang-liu/phishing-email-dataset")
print(f"Splits found: {list(ds_main.keys())}")

# Convert Hugging Face train split to Polars
df_main = pl.from_arrow(ds_main['train'].data.table)
print(f"Main Dataset Shape: {df_main.shape}")
print(f"Columns: {df_main.columns}")
print(df_main.head(2))


print("\n" + "="*60)
print("2. LOADING OUT-OF-DISTRIBUTION DATASET (Kaggle: Naser Abdullah Alam)")
print("="*60)
# Look for the dataset in Kaggle input or download via kagglehub
ood_files = glob.glob("/kaggle/input/**/phishing-email-dataset*/**/*.csv", recursive=True) + \
            glob.glob("/kaggle/input/**/Phishing_Email.csv", recursive=True)

if not ood_files:
    print("Downloading via kagglehub...")
    ood_path = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")
    ood_files = glob.glob(f"{ood_path}/**/*.csv", recursive=True)

print(f"Found OOD files: {ood_files}")
if ood_files:
    df_ood = pl.read_csv(ood_files[0], ignore_errors=True)
    print(f"OOD Dataset Shape: {df_ood.shape}")
    print(f"Columns: {df_ood.columns}")
    print(df_ood.head(2))


print("\n" + "="*60)
print("3. LOADING SMS SLICE DATASET (Kaggle: SMS Spam Collection)")
print("="*60)
sms_files = glob.glob("/kaggle/input/**/spam.csv", recursive=True)
if not sms_files:
    print("Downloading SMS dataset via kagglehub...")
    sms_path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
    sms_files = glob.glob(f"{sms_path}/**/*.csv", recursive=True)

print(f"Found SMS files: {sms_files}")
if sms_files:
    # SMS dataset typically has latin-1 encoding
    df_sms = pl.read_csv(sms_files[0], encoding="latin1", ignore_errors=True)
    print(f"SMS Dataset Shape: {df_sms.shape}")
    print(f"Columns: {df_sms.columns}")
    print(df_sms.head(2))


print("\n" + "="*60)
print("4. LOADING AI-GENERATED PHISHING (Hugging Face: premsaidhulipala)")
print("="*60)
try:
    ds_ai = load_dataset("premsaidhulipala/ai-phishing-dataset")
    df_ai = pl.from_arrow(ds_ai['train'].data.table)
    print(f"AI Phishing Shape: {df_ai.shape}")
    print(f"Columns: {df_ai.columns}")
    print(df_ai.head(2))
except Exception as e:
    print(f"Could not load AI dataset directly: {e}")

1. LOADING MAIN TRAINING DATASET (Hugging Face: MeAJOR)


README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

Splits found: ['train']
Main Dataset Shape: (18650, 3)
Columns: ['Unnamed: 0', 'Email Text', 'Email Type']
shape: (2, 3)
┌────────────┬─────────────────────────────────┬────────────┐
│ Unnamed: 0 ┆ Email Text                      ┆ Email Type │
│ ---        ┆ ---                             ┆ ---        │
│ i64        ┆ str                             ┆ str        │
╞════════════╪═════════════════════════════════╪════════════╡
│ 0          ┆ re : 6 . 1100 , disc : uniform… ┆ Safe Email │
│ 1          ┆ the other side of * galicismos… ┆ Safe Email │
└────────────┴─────────────────────────────────┴────────────┘

2. LOADING OUT-OF-DISTRIBUTION DATASET (Kaggle: Naser Abdullah Alam)
Found OOD files: ['/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/SpamAssasin.csv', '/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/Nazario.csv', '/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/Nigerian_Fraud.csv', '/kaggle/input/datasets/naserabdullahalam/phish

In [3]:
from huggingface_hub import HfApi

api = HfApi()
results = list(api.list_datasets(search="ai phishing", limit=10))
print(f"Found {len(results)} matching datasets:")
for r in results:
    print(f"- {r.id}")

Found 3 matching datasets:
- lleratodev/ai-powered-phishing-email-detection-system
- premsaid/ai-phishing-dataset
- cont1n3nt/ai-phishing-dataset


In [4]:
from datasets import load_dataset
import polars as pl

print("Loading premsaid/ai-phishing-dataset...")
ds_ai = load_dataset("premsaid/ai-phishing-dataset")
print("Splits:", list(ds_ai.keys()))

# Check first split structure
split_name = list(ds_ai.keys())[0]
df_ai = pl.from_arrow(ds_ai[split_name].data.table)
print(f"AI Phishing Dataset Shape: {df_ai.shape}")
print(f"Columns: {df_ai.columns}")
print(df_ai.head(3))

Loading premsaid/ai-phishing-dataset...


README.md:   0%|          | 0.00/391 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


final_ai_phish_dataset.csv: reconstructing file:   0%|          |  0.00B / 52.1MB            

final_ai_phish_dataset.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18655 [00:00<?, ? examples/s]

Splits: ['train']
AI Phishing Dataset Shape: (18655, 4)
Columns: ['Unnamed: 0', 'Email Text', 'Email Type', 'label']
shape: (3, 4)
┌────────────┬─────────────────────────────────┬────────────┬───────┐
│ Unnamed: 0 ┆ Email Text                      ┆ Email Type ┆ label │
│ ---        ┆ ---                             ┆ ---        ┆ ---   │
│ i64        ┆ str                             ┆ str        ┆ i64   │
╞════════════╪═════════════════════════════════╪════════════╪═══════╡
│ 0          ┆ re : 6 . 1100 , disc : uniform… ┆ Safe Email ┆ 0     │
│ 1          ┆ the other side of * galicismos… ┆ Safe Email ┆ 0     │
│ 2          ┆ re : equistar deal tickets are… ┆ Safe Email ┆ 0     │
└────────────┴─────────────────────────────────┴────────────┴───────┘


In [1]:
# Cell 1: Environment Setup & Library Installations
!pip install -q polars pyarrow datasets kagglehub scikit-learn xgboost shap transformers accelerate

import sys
import polars as pl
import sklearn
import torch

print("=" * 60)
print("CELL 1 COMPLETE: ENVIRONMENT VERIFIED")
print(f"Python version : {sys.version.split()[0]}")
print(f"Polars version : {pl.__version__}")
print(f"PyTorch version: {torch.__version__} (CUDA available: {torch.cuda.is_available()})")
if torch.cuda.is_available():
    print(f"Device name    : {torch.cuda.get_device_name(0)}")
print("=" * 60)

CELL 1 COMPLETE: ENVIRONMENT VERIFIED
Python version : 3.13.15
Polars version : 1.35.2
PyTorch version: 2.11.0+cu128 (CUDA available: True)
Device name    : Tesla T4


In [2]:
# Cell 2: Project Scaffold & Directory Creation
import os

# Create repository directories
directories = [
    "data/raw",
    "data/processed",
    "reports/figures",
    "src",
    "models",
    "tests"
]

for d in directories:
    os.makedirs(f"/kaggle/working/{d}", exist_ok=True)

print("=" * 60)
print("CELL 2 COMPLETE: DIRECTORY SCAFFOLD CREATED")
for d in directories:
    print(f"  [✓] /kaggle/working/{d}")
print("=" * 60)

CELL 2 COMPLETE: DIRECTORY SCAFFOLD CREATED
  [✓] /kaggle/working/data/raw
  [✓] /kaggle/working/data/processed
  [✓] /kaggle/working/reports/figures
  [✓] /kaggle/working/src
  [✓] /kaggle/working/models
  [✓] /kaggle/working/tests


In [3]:
# Cell 3: Raw Data Ingestion & Schema Inspection
import glob
import kagglehub
import polars as pl
from datasets import load_dataset

print("=" * 60)
print("1. INSPECTING MAIN DATASET (Hugging Face: zefang-liu)")
print("=" * 60)
ds_main_hf = load_dataset("zefang-liu/phishing-email-dataset")
df_main_raw = pl.from_arrow(ds_main_hf["train"].data.table)
print(f"Main Shape: {df_main_raw.shape}")
print(f"Main Columns: {df_main_raw.columns}")
print(df_main_raw.head(2))

print("\n" + "=" * 60)
print("2. INSPECTING OOD EMAIL DATASET (Kaggle: Naser Abdullah Alam)")
print("=" * 60)
ood_paths = glob.glob("/kaggle/input/**/SpamAssasin.csv", recursive=True)
if not ood_paths:
    print("Downloading OOD dataset via kagglehub...")
    download_dir = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")
    ood_paths = glob.glob(f"{download_dir}/**/*.csv", recursive=True)
else:
    ood_dir = os.path.dirname(ood_paths[0])
    ood_paths = glob.glob(f"{ood_dir}/*.csv")

print(f"Found {len(ood_paths)} CSV files in OOD dataset:")
for p in ood_paths:
    sample_df = pl.read_csv(p, n_rows=2, ignore_errors=True)
    print(f"  - {os.path.basename(p)}: Columns = {sample_df.columns}")

print("\n" + "=" * 60)
print("3. INSPECTING SMS DATASET (Kaggle: SMS Spam Collection)")
print("=" * 60)
sms_paths = glob.glob("/kaggle/input/**/spam.csv", recursive=True)
if not sms_paths:
    print("Downloading SMS dataset via kagglehub...")
    sms_dir = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
    sms_paths = glob.glob(f"{sms_dir}/**/*.csv", recursive=True)

df_sms_raw = pl.read_csv(sms_paths[0], encoding="latin1", n_rows=2, ignore_errors=True)
print(f"SMS File: {sms_paths[0]}")
print(f"SMS Columns: {df_sms_raw.columns}")
print(df_sms_raw.head(2))

print("\nCELL 3 COMPLETE: RAW DATA SOURCES DISCOVERED & INSPECTED")

1. INSPECTING MAIN DATASET (Hugging Face: zefang-liu)


README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

Main Shape: (18650, 3)
Main Columns: ['Unnamed: 0', 'Email Text', 'Email Type']
shape: (2, 3)
┌────────────┬─────────────────────────────────┬────────────┐
│ Unnamed: 0 ┆ Email Text                      ┆ Email Type │
│ ---        ┆ ---                             ┆ ---        │
│ i64        ┆ str                             ┆ str        │
╞════════════╪═════════════════════════════════╪════════════╡
│ 0          ┆ re : 6 . 1100 , disc : uniform… ┆ Safe Email │
│ 1          ┆ the other side of * galicismos… ┆ Safe Email │
└────────────┴─────────────────────────────────┴────────────┘

2. INSPECTING OOD EMAIL DATASET (Kaggle: Naser Abdullah Alam)
Found 7 CSV files in OOD dataset:
  - SpamAssasin.csv: Columns = ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
  - Nazario.csv: Columns = ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
  - Nigerian_Fraud.csv: Columns = ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
  - CEAS_08.csv: Col

In [4]:
# Cell 4: Implement src/data.py
data_py_code = '''"""
src/data.py
Data processing, cleaning, deduplication, and leak-free splitting.
"""
import os
import re
import glob
from typing import Tuple, Dict, Any
import polars as pl
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import kagglehub


def clean_and_normalize_text(text: str) -> str:
    """Strips HTML tags and normalizes consecutive whitespace."""
    if not text:
        return ""
    # Strip HTML tags
    cleaned = re.sub(r"<[^>]+>", " ", text)
    # Normalize whitespaces
    cleaned = re.sub(r"\\s+", " ", cleaned).strip()
    return cleaned


def create_dedup_key(text: str) -> str:
    """Creates a normalized lowercase string key for zero-leakage deduplication."""
    return re.sub(r"[^a-zA-Z0-9]", "", text.lower())


def load_and_clean_main_dataset() -> pl.DataFrame:
    """Loads HF zefang-liu/phishing-email-dataset and cleans it."""
    print("Loading Hugging Face main dataset...")
    ds = load_dataset("zefang-liu/phishing-email-dataset")
    df_raw = pl.from_arrow(ds["train"].data.table)

    df_cleaned = (
        df_raw
        .filter(pl.col("Email Text").is_not_null())
        .with_columns([
            pl.col("Email Text").map_elements(clean_and_normalize_text, return_dtype=pl.String).alias("text"),
            pl.when(pl.col("Email Type").str.to_lowercase().str.contains("phishing"))
              .then(1)
              .otherwise(0)
              .cast(pl.Int32)
              .alias("label"),
            pl.lit("main_email").alias("source")
        ])
        .filter(pl.col("text").str.len_chars() > 10)
        .with_columns([
            pl.col("text").map_elements(create_dedup_key, return_dtype=pl.String).alias("dedup_key")
        ])
        .unique(subset=["dedup_key"])
        .select(["text", "label", "source", "dedup_key"])
    )
    return df_cleaned


def load_and_clean_ood_dataset() -> pl.DataFrame:
    """Loads Kaggle naserabdullahalam/phishing-email-dataset CSVs and cleans them."""
    print("Loading Kaggle OOD email dataset...")
    ood_paths = glob.glob("/kaggle/input/**/SpamAssasin.csv", recursive=True)
    if not ood_paths:
        download_dir = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")
        ood_dir = download_dir
    else:
        ood_dir = os.path.dirname(ood_paths[0])

    csv_files = glob.glob(f"{ood_dir}/*.csv")
    ood_dfs = []

    for f in csv_files:
        base = os.path.basename(f)
        if base == "phishing_email.csv":
            continue  # Skip merged file to avoid duplicate counting of sub-sources
        source_name = f"ood_email_{base.replace('.csv', '').lower()}"
        try:
            df_temp = pl.read_csv(f, ignore_errors=True)
            if "body" in df_temp.columns and "label" in df_temp.columns:
                if "subject" in df_temp.columns:
                    df_temp = df_temp.with_columns(
                        (pl.col("subject").fill_null("") + " " + pl.col("body").fill_null("")).alias("raw_text")
                    )
                else:
                    df_temp = df_temp.with_columns(pl.col("body").alias("raw_text"))

                cleaned = (
                    df_temp
                    .filter(pl.col("raw_text").is_not_null())
                    .with_columns([
                        pl.col("raw_text").map_elements(clean_and_normalize_text, return_dtype=pl.String).alias("text"),
                        pl.col("label").cast(pl.Int32).alias("label"),
                        pl.lit(source_name).alias("source")
                    ])
                    .filter(pl.col("text").str.len_chars() > 10)
                    .with_columns([
                        pl.col("text").map_elements(create_dedup_key, return_dtype=pl.String).alias("dedup_key")
                    ])
                    .unique(subset=["dedup_key"])
                    .select(["text", "label", "source", "dedup_key"])
                )
                ood_dfs.append(cleaned)
        except Exception as e:
            print(f"Error loading {base}: {e}")

    return pl.concat(ood_dfs).unique(subset=["dedup_key"])


def load_and_clean_sms_dataset() -> pl.DataFrame:
    """Loads Kaggle SMS dataset and cleans it."""
    print("Loading Kaggle SMS dataset...")
    sms_paths = glob.glob("/kaggle/input/**/spam.csv", recursive=True)
    if not sms_paths:
        download_dir = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
        sms_file = glob.glob(f"{download_dir}/**/*.csv", recursive=True)[0]
    else:
        sms_file = sms_paths[0]

    df_raw = pl.read_csv(sms_file, encoding="latin1", ignore_errors=True)
    cleaned = (
        df_raw
        .filter(pl.col("v2").is_not_null())
        .with_columns([
            pl.col("v2").map_elements(clean_and_normalize_text, return_dtype=pl.String).alias("text"),
            pl.when(pl.col("v1").str.to_lowercase() == "spam")
              .then(1)
              .otherwise(0)
              .cast(pl.Int32)
              .alias("label"),
            pl.lit("sms_smishing").alias("source")
        ])
        .filter(pl.col("text").str.len_chars() > 5)
        .with_columns([
            pl.col("text").map_elements(create_dedup_key, return_dtype=pl.String).alias("dedup_key")
        ])
        .unique(subset=["dedup_key"])
        .select(["text", "label", "source", "dedup_key"])
    )
    return cleaned


def run_pipeline(output_dir: str = "/kaggle/working/data/processed", reports_dir: str = "/kaggle/working/reports") -> Dict[str, Any]:
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(reports_dir, exist_ok=True)

    # 1. Load datasets
    df_main = load_and_clean_main_dataset()
    df_ood_raw = load_and_clean_ood_dataset()
    df_sms_raw = load_and_clean_sms_dataset()

    # 2. Cross-Dataset Deduplication (Ensure NO overlap between Main and OOD/SMS)
    main_keys = set(df_main["dedup_key"].to_list())
    df_ood = df_ood_raw.filter(~pl.col("dedup_key").is_in(main_keys))
    df_sms = df_sms_raw.filter(~pl.col("dedup_key").is_in(main_keys))

    ood_dropped = df_ood_raw.height - df_ood.height
    sms_dropped = df_sms_raw.height - df_sms.height
    print(f"Zero-Leakage Filter: Dropped {ood_dropped} overlapping emails from OOD dataset.")
    print(f"Zero-Leakage Filter: Dropped {sms_dropped} overlapping texts from SMS dataset.")

    # 3. Stratified Train / Val / Test split on Main (70 / 15 / 15, seed 42)
    pdf = df_main.to_pandas()
    train_val, test = train_test_split(pdf, test_size=0.15, random_state=42, stratify=pdf["label"])
    train, val = train_test_split(train_val, test_size=0.1765, random_state=42, stratify=train_val["label"])

    df_train = pl.from_pandas(train).select(["text", "label", "source"])
    df_val = pl.from_pandas(val).select(["text", "label", "source"])
    df_test = pl.from_pandas(test).select(["text", "label", "source"])
    df_ood_final = df_ood.select(["text", "label", "source"])
    df_sms_final = df_sms.select(["text", "label", "source"])

    # 4. Save to Parquet
    paths = {
        "main_train": f"{output_dir}/main_train.parquet",
        "main_val": f"{output_dir}/main_val.parquet",
        "main_test": f"{output_dir}/main_test.parquet",
        "ood_email": f"{output_dir}/ood_email.parquet",
        "sms_test": f"{output_dir}/sms_test.parquet",
    }
    df_train.write_parquet(paths["main_train"])
    df_val.write_parquet(paths["main_val"])
    df_test.write_parquet(paths["main_test"])
    df_ood_final.write_parquet(paths["ood_email"])
    df_sms_final.write_parquet(paths["sms_test"])

    # 5. Generate data_report.md
    report_content = f"""# Data Ingestion & Cleaning Report

## Split Summary & Class Distribution
| Split | Total Rows | Phishing / Spam (1) | Safe / Ham (0) | Pos Class Ratio |
| :--- | :--- | :--- | :--- | :--- |
| **Main Train (70%)** | {df_train.height:,} | {df_train['label'].sum():,} | {df_train.height - df_train['label'].sum():,} | {df_train['label'].mean():.2%} |
| **Main Val (15%)** | {df_val.height:,} | {df_val['label'].sum():,} | {df_val.height - df_val['label'].sum():,} | {df_val['label'].mean():.2%} |
| **Main Test (15%)** | {df_test.height:,} | {df_test['label'].sum():,} | {df_test.height - df_test['label'].sum():,} | {df_test['label'].mean():.2%} |
| **OOD Email Test** | {df_ood_final.height:,} | {df_ood_final['label'].sum():,} | {df_ood_final.height - df_ood_final['label'].sum():,} | {df_ood_final['label'].mean():.2%} |
| **SMS Test (Channel Shift)** | {df_sms_final.height:,} | {df_sms_final['label'].sum():,} | {df_sms_final.height - df_sms_final['label'].sum():,} | {df_sms_final['label'].mean():.2%} |

## Zero-Leakage & Deduplication Safeguards
- **Main Dataset Deduplication**: Deduplicated using normalized alphanumeric keys before any split.
- **OOD Cross-Deduplication**: {ood_dropped:,} samples from the OOD corpus were dropped because they were near-duplicates of Main training emails.
- **SMS Cross-Deduplication**: {sms_dropped:,} samples dropped for overlapping keys.
- **Random Seed**: Fixed at 42 across all splits.
"""
    with open(f"{reports_dir}/data_report.md", "w") as rf:
        rf.write(report_content)

    print("Data processing complete! Parquets saved and reports/data_report.md created.")
    return paths

if __name__ == "__main__":
    run_pipeline()
'''

with open("/kaggle/working/src/data.py", "w") as f:
    f.write(data_py_code)

print("Saved /kaggle/working/src/data.py successfully!")

# Run the pipeline immediately
import sys
sys.path.append("/kaggle/working")
from src.data import run_pipeline

paths = run_pipeline()

Saved /kaggle/working/src/data.py successfully!
Loading Hugging Face main dataset...
Loading Kaggle OOD email dataset...
Loading Kaggle SMS dataset...
Zero-Leakage Filter: Dropped 12017 overlapping emails from OOD dataset.
Zero-Leakage Filter: Dropped 0 overlapping texts from SMS dataset.
Data processing complete! Parquets saved and reports/data_report.md created.


In [5]:
# Cell 5: tests/test_data.py & Running Pytest
test_data_code = '''"""
tests/test_data.py
Asserts zero-leakage and schema integrity across all processed parquet datasets.
"""
import polars as pl
import pytest
import re

DATA_DIR = "/kaggle/working/data/processed"

def get_norm_set(df: pl.DataFrame) -> set:
    return set(df["text"].map_elements(lambda t: re.sub(r"[^a-zA-Z0-9]", "", t.lower()), return_dtype=pl.String).to_list())

def test_parquet_files_exist():
    import os
    for fname in ["main_train.parquet", "main_val.parquet", "main_test.parquet", "ood_email.parquet", "sms_test.parquet"]:
        path = f"{DATA_DIR}/{fname}"
        assert os.path.exists(path), f"Missing {path}"

def test_zero_leakage_between_in_domain_splits():
    train_df = pl.read_parquet(f"{DATA_DIR}/main_train.parquet")
    val_df = pl.read_parquet(f"{DATA_DIR}/main_val.parquet")
    test_df = pl.read_parquet(f"{DATA_DIR}/main_test.parquet")

    train_set = get_norm_set(train_df)
    val_set = get_norm_set(val_df)
    test_set = get_norm_set(test_df)

    assert len(train_set.intersection(val_set)) == 0, "Leakage found between Train and Val!"
    assert len(train_set.intersection(test_set)) == 0, "Leakage found between Train and Test!"
    assert len(val_set.intersection(test_set)) == 0, "Leakage found between Val and Test!"

def test_zero_leakage_to_ood_and_sms():
    train_df = pl.read_parquet(f"{DATA_DIR}/main_train.parquet")
    ood_df = pl.read_parquet(f"{DATA_DIR}/ood_email.parquet")
    sms_df = pl.read_parquet(f"{DATA_DIR}/sms_test.parquet")

    train_set = get_norm_set(train_df)
    ood_set = get_norm_set(ood_df)
    sms_set = get_norm_set(sms_df)

    assert len(train_set.intersection(ood_set)) == 0, "Leakage found between Train and OOD Email!"
    assert len(train_set.intersection(sms_set)) == 0, "Leakage found between Train and SMS Test!"

def test_labels_are_binary():
    for fname in ["main_train.parquet", "main_val.parquet", "main_test.parquet", "ood_email.parquet", "sms_test.parquet"]:
        df = pl.read_parquet(f"{DATA_DIR}/{fname}")
        labels = set(df["label"].unique().to_list())
        assert labels.issubset({0, 1}), f"Invalid labels in {fname}: {labels}"
        assert df["label"].sum() > 0, f"No positive class examples in {fname}!"
'''

with open("/kaggle/working/tests/test_data.py", "w") as f:
    f.write(test_data_code)

print("Saved /kaggle/working/tests/test_data.py!")

# Run pytest on the test suite
!pytest -v /kaggle/working/tests/test_data.py

Saved /kaggle/working/tests/test_data.py!
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 4 items                                                              

tests/test_data.py::test_parquet_files_exist PASSED                      [ 25%]
tests/test_data.py::test_zero_leakage_between_in_domain_splits PASSED    [ 50%]
tests/test_data.py::test_zero_leakage_to_ood_and_sms PASSED              [ 75%]
tests/test_data.py::test_labels_are_binary PASSED                        [100%]

============================== 4 passed in 13.83s ==============================


In [6]:
# Cell 6: Implement src/evaluate.py
evaluate_py_code = '''"""
src/evaluate.py
Comprehensive evaluation metrics, curve plotting, and results logging.
"""
import os
import csv
from typing import Dict, Any, Optional
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)


def compute_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float = 0.5
) -> Dict[str, float]:
    """Computes binary classification metrics given probabilities and a decision threshold."""
    y_pred = (y_prob >= threshold).astype(int)
    
    # Handle single-class edge cases gracefully
    try:
        roc_auc = float(roc_auc_score(y_true, y_prob))
    except Exception:
        roc_auc = 0.0

    try:
        pr_auc = float(average_precision_score(y_true, y_prob))
    except Exception:
        pr_auc = 0.0

    return {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "threshold": float(threshold)
    }


def evaluate_and_log(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    model_name: str,
    split_name: str,
    threshold: float = 0.5,
    reports_dir: str = "/kaggle/working/reports",
    save_plots: bool = True
) -> Dict[str, float]:
    """Computes metrics, plots curves, and appends to reports/results.csv."""
    os.makedirs(reports_dir, exist_ok=True)
    fig_dir = os.path.join(reports_dir, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    metrics = compute_metrics(y_true, y_prob, threshold=threshold)
    y_pred = (y_prob >= threshold).astype(int)

    # 1. Plot & Save PR Curve
    if save_plots:
        plt.figure(figsize=(6, 5))
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        plt.plot(rec, prec, label=f"PR-AUC = {metrics['pr_auc']:.4f}", color="#1f77b4", lw=2)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"PR Curve: {model_name} ({split_name})")
        plt.legend(loc="lower left")
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()
        plt.savefig(f"{fig_dir}/pr_curve_{model_name}_{split_name}.png", dpi=150)
        plt.close()

        # 2. Plot & Save Confusion Matrix
        plt.figure(figsize=(5, 4))
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit", "Phish"])
        disp.plot(cmap="Blues", values_format="d")
        plt.title(f"CM: {model_name} ({split_name}) @ {threshold:.2f}")
        plt.tight_layout()
        plt.savefig(f"{fig_dir}/cm_{model_name}_{split_name}.png", dpi=150)
        plt.close()

    # 3. Append to reports/results.csv
    csv_path = os.path.join(reports_dir, "results.csv")
    file_exists = os.path.exists(csv_path)

    fieldnames = ["model", "split", "precision", "recall", "f1", "roc_auc", "pr_auc", "threshold"]
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            "model": model_name,
            "split": split_name,
            "precision": round(metrics["precision"], 4),
            "recall": round(metrics["recall"], 4),
            "f1": round(metrics["f1"], 4),
            "roc_auc": round(metrics["roc_auc"], 4),
            "pr_auc": round(metrics["pr_auc"], 4),
            "threshold": round(metrics["threshold"], 4)
        })

    print(f"[{model_name}] Split: {split_name:<12} | PR-AUC: {metrics['pr_auc']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f} | F1: {metrics['f1']:.4f} | Prec: {metrics['precision']:.4f} | Rec: {metrics['recall']:.4f}")
    return metrics
'''

with open("/kaggle/working/src/evaluate.py", "w") as f:
    f.write(evaluate_py_code)

print("Saved /kaggle/working/src/evaluate.py successfully!")

Saved /kaggle/working/src/evaluate.py successfully!


In [7]:
# Cell 7: Train & Evaluate Baseline (TF-IDF + Logistic Regression)
import sys
sys.path.append("/kaggle/working")

import numpy as np
import polars as pl
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from src.evaluate import evaluate_and_log

print("=" * 60)
print("TRAINING BASELINE: TF-IDF + LOGISTIC REGRESSION")
print("=" * 60)

DATA_DIR = "/kaggle/working/data/processed"

# 1. Load Parquets
train_df = pl.read_parquet(f"{DATA_DIR}/main_train.parquet")
val_df = pl.read_parquet(f"{DATA_DIR}/main_val.parquet")
test_df = pl.read_parquet(f"{DATA_DIR}/main_test.parquet")
ood_df = pl.read_parquet(f"{DATA_DIR}/ood_email.parquet")
sms_df = pl.read_parquet(f"{DATA_DIR}/sms_test.parquet")

X_train, y_train = train_df["text"].to_list(), train_df["label"].to_numpy()
X_val, y_val = val_df["text"].to_list(), val_df["label"].to_numpy()
X_test, y_test = test_df["text"].to_list(), test_df["label"].to_numpy()
X_ood, y_ood = ood_df["text"].to_list(), ood_df["label"].to_numpy()
X_sms, y_sms = sms_df["text"].to_list(), sms_df["label"].to_numpy()

# 2. Build Pipeline
baseline_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=25000, ngram_range=(1, 2), sublinear_tf=True)),
    ("clf", LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

print("Fitting TF-IDF + Logistic Regression on Train...")
baseline_pipe.fit(X_train, y_train)

# 3. Predict Probabilities (Positive class)
val_probs = baseline_pipe.predict_proba(X_val)[:, 1]
test_probs = baseline_pipe.predict_proba(X_test)[:, 1]
ood_probs = baseline_pipe.predict_proba(X_ood)[:, 1]
sms_probs = baseline_pipe.predict_proba(X_sms)[:, 1]

# 4. Evaluate across all splits & log to reports/results.csv
print("\n--- Baseline Results ---")
evaluate_and_log(y_val, val_probs, model_name="tfidf_lr", split_name="val")
evaluate_and_log(y_test, test_probs, model_name="tfidf_lr", split_name="test")
evaluate_and_log(y_ood, ood_probs, model_name="tfidf_lr", split_name="ood_email")
evaluate_and_log(y_sms, sms_probs, model_name="tfidf_lr", split_name="sms_test")

# 5. Save model and probabilities for downstream stacking
joblib.dump(baseline_pipe, "/kaggle/working/models/baseline_tfidf_lr.joblib")
np.save("/kaggle/working/models/val_preds_tfidf.npy", val_probs)
np.save("/kaggle/working/models/test_preds_tfidf.npy", test_probs)
np.save("/kaggle/working/models/ood_preds_tfidf.npy", ood_probs)
np.save("/kaggle/working/models/sms_preds_tfidf.npy", sms_probs)

print("\nCELL 7 COMPLETE: BASELINE TRAINED & LOGGED")

TRAINING BASELINE: TF-IDF + LOGISTIC REGRESSION
Fitting TF-IDF + Logistic Regression on Train...

--- Baseline Results ---
[tfidf_lr] Split: val          | PR-AUC: 0.9961 | ROC-AUC: 0.9978 | F1: 0.9689 | Prec: 0.9801 | Rec: 0.9579
[tfidf_lr] Split: test         | PR-AUC: 0.9954 | ROC-AUC: 0.9975 | F1: 0.9732 | Prec: 0.9762 | Rec: 0.9702
[tfidf_lr] Split: ood_email    | PR-AUC: 0.9753 | ROC-AUC: 0.9800 | F1: 0.9142 | Prec: 0.9538 | Rec: 0.8779
[tfidf_lr] Split: sms_test     | PR-AUC: 0.3394 | ROC-AUC: 0.7882 | F1: 0.3175 | Prec: 0.1935 | Rec: 0.8834

CELL 7 COMPLETE: BASELINE TRAINED & LOGGED


<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

In [8]:
# Cell 8: Implement src/features.py
features_py_code = '''"""
src/features.py
Handcrafted cybersecurity risk signals and feature extraction using Polars and Regex.
"""
import re
from typing import List, Dict, Any
import numpy as np
import polars as pl

# Common URL shortener domains used in social engineering
SHORTENERS = {"bit.ly", "tinyurl.com", "t.co", "is.gd", "buff.ly", "ow.ly", "goo.gl", "cutt.ly"}

# High-risk urgency and social engineering keywords
URGENCY_KEYWORDS = [
    r"\\bverify\\b", r"\\bsuspended\\b", r"\\bimmediately\\b", r"\\burgent\\b",
    r"\\bpassword\\b", r"\\botp\\b", r"\\bkyc\\b", r"\\bblocked\\b",
    r"\\bsecurity alert\\b", r"\\bunauthorized\\b", r"\\baction required\\b",
    r"\\bexpire\\b", r"\\blogin\\b", r"\\baccount\\b", r"\\bbank\\b"
]
URGENCY_REGEX = re.compile("|".join(URGENCY_KEYWORDS), re.IGNORECASE)
URL_REGEX = re.compile(r"https?://[^\\s]+|www\\.[^\\s]+", re.IGNORECASE)
IP_IN_URL_REGEX = re.compile(r"https?://(?:\\d{1,3}\\.){3}\\d{1,3}")


def extract_features_single(text: str) -> Dict[str, float]:
    """Extracts domain-informed cybersecurity features from a single raw text."""
    if not text:
        text = ""
        
    length = len(text)
    urls = URL_REGEX.findall(text)
    num_urls = len(urls)
    max_url_len = max([len(u) for u in urls], default=0)
    has_ip_url = int(bool(IP_IN_URL_REGEX.search(text)))
    
    # URL character obfuscation features
    at_in_urls = sum(u.count("@") for u in urls)
    hyphen_in_urls = sum(u.count("-") for u in urls)
    
    # URL shortener detection
    has_shortener = int(any(any(s in u.lower() for s in SHORTENERS) for u in urls))
    
    # Text-level lexical & urgency features
    digit_count = sum(c.isdigit() for c in text)
    upper_count = sum(c.isupper() for c in text)
    excl_count = text.count("!")
    qmark_count = text.count("?")
    
    digit_ratio = (digit_count / length) if length > 0 else 0.0
    upper_ratio = (upper_count / length) if length > 0 else 0.0
    urgency_hits = len(URGENCY_REGEX.findall(text))

    return {
        "num_urls": float(num_urls),
        "max_url_len": float(max_url_len),
        "has_ip_url": float(has_ip_url),
        "at_in_urls": float(at_in_urls),
        "hyphen_in_urls": float(hyphen_in_urls),
        "has_shortener": float(has_shortener),
        "digit_ratio": float(digit_ratio),
        "upper_ratio": float(upper_ratio),
        "excl_count": float(excl_count),
        "qmark_count": float(qmark_count),
        "urgency_hits": float(urgency_hits),
        "text_length": float(length)
    }


def extract_feature_matrix(df: pl.DataFrame) -> np.ndarray:
    """Batch-extracts features for a Polars DataFrame using vectorized expressions."""
    texts = df["text"].to_list()
    feature_dicts = [extract_features_single(t) for t in texts]
    # Return as structured float32 numpy matrix
    return np.array([[d[k] for k in FEATURE_NAMES] for d in feature_dicts], dtype=np.float32)


FEATURE_NAMES = [
    "num_urls",
    "max_url_len",
    "has_ip_url",
    "at_in_urls",
    "hyphen_in_urls",
    "has_shortener",
    "digit_ratio",
    "upper_ratio",
    "excl_count",
    "qmark_count",
    "urgency_hits",
    "text_length"
]
'''

with open("/kaggle/working/src/features.py", "w") as f:
    f.write(features_py_code)

print("Saved /kaggle/working/src/features.py successfully!")

Saved /kaggle/working/src/features.py successfully!


In [9]:
# Cell 9: Train XGBoost Feature Model, 5-Fold CV & SHAP
import sys
sys.path.append("/kaggle/working")

import numpy as np
import polars as pl
import joblib
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import shap

from src.features import extract_feature_matrix, FEATURE_NAMES
from src.evaluate import evaluate_and_log

print("=" * 60)
print("TRAINING TABULAR FEATURE MODEL (XGBOOST)")
print("=" * 60)

DATA_DIR = "/kaggle/working/data/processed"

# 1. Load splits
train_df = pl.read_parquet(f"{DATA_DIR}/main_train.parquet")
val_df = pl.read_parquet(f"{DATA_DIR}/main_val.parquet")
test_df = pl.read_parquet(f"{DATA_DIR}/main_test.parquet")
ood_df = pl.read_parquet(f"{DATA_DIR}/ood_email.parquet")
sms_df = pl.read_parquet(f"{DATA_DIR}/sms_test.parquet")

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()
y_ood = ood_df["label"].to_numpy()
y_sms = sms_df["label"].to_numpy()

# 2. Extract feature matrices
print("Extracting security features from all splits...")
X_train_feats = extract_feature_matrix(train_df)
X_val_feats = extract_feature_matrix(val_df)
X_test_feats = extract_feature_matrix(test_df)
X_ood_feats = extract_feature_matrix(ood_df)
X_sms_feats = extract_feature_matrix(sms_df)
print(f"Feature matrix shape: {X_train_feats.shape} across {len(FEATURE_NAMES)} signals.")

# 3. 5-Fold Stratified Cross-Validation on Train
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

print("\nRunning 5-Fold Stratified CV on Training Set...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_train_feats, y_train, cv=cv, scoring="average_precision")
print(f"5-Fold CV PR-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# 4. Fit Model on full train set
print("\nFitting final XGBoost model...")
xgb_model.fit(X_train_feats, y_train)

# 5. Predict Probabilities
val_probs = xgb_model.predict_proba(X_val_feats)[:, 1]
test_probs = xgb_model.predict_proba(X_test_feats)[:, 1]
ood_probs = xgb_model.predict_proba(X_ood_feats)[:, 1]
sms_probs = xgb_model.predict_proba(X_sms_feats)[:, 1]

# 6. Evaluate & Log to reports/results.csv
print("\n--- XGBoost Feature Model Results ---")
evaluate_and_log(y_val, val_probs, model_name="xgb_features", split_name="val")
evaluate_and_log(y_test, test_probs, model_name="xgb_features", split_name="test")
evaluate_and_log(y_ood, ood_probs, model_name="xgb_features", split_name="ood_email")
evaluate_and_log(y_sms, sms_probs, model_name="xgb_features", split_name="sms_test")

# 7. Compute SHAP Values on sample & save plot
print("\nComputing SHAP summary values for interpretability...")
explainer = shap.TreeExplainer(xgb_model)
sample_idx = np.random.choice(len(X_test_feats), size=min(500, len(X_test_feats)), replace=False)
shap_values = explainer.shap_values(X_test_feats[sample_idx])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_feats[sample_idx], feature_names=FEATURE_NAMES, show=False)
plt.title("SHAP Feature Importance: Security Risk Signals", fontsize=14)
plt.tight_layout()
plt.savefig("/kaggle/working/reports/figures/shap_summary.png", dpi=150)
plt.close()
print("Saved SHAP summary plot to reports/figures/shap_summary.png")

# 8. Save Model and probabilities
joblib.dump(xgb_model, "/kaggle/working/models/xgb_features.joblib")
np.save("/kaggle/working/models/val_preds_xgb.npy", val_probs)
np.save("/kaggle/working/models/test_preds_xgb.npy", test_probs)
np.save("/kaggle/working/models/ood_preds_xgb.npy", ood_probs)
np.save("/kaggle/working/models/sms_preds_xgb.npy", sms_probs)

print("\nCELL 9 COMPLETE: XGBOOST MODEL & SHAP EXPLANATIONS LOGGED")

TRAINING TABULAR FEATURE MODEL (XGBOOST)
Extracting security features from all splits...
Feature matrix shape: (12224, 12) across 12 signals.

Running 5-Fold Stratified CV on Training Set...
5-Fold CV PR-AUC: 0.8373 (+/- 0.0123)

Fitting final XGBoost model...

--- XGBoost Feature Model Results ---
[xgb_features] Split: val          | PR-AUC: 0.8286 | ROC-AUC: 0.8896 | F1: 0.7366 | Prec: 0.7922 | Rec: 0.6882
[xgb_features] Split: test         | PR-AUC: 0.8326 | ROC-AUC: 0.8884 | F1: 0.7265 | Prec: 0.8000 | Rec: 0.6653
[xgb_features] Split: ood_email    | PR-AUC: 0.8148 | ROC-AUC: 0.8059 | F1: 0.6153 | Prec: 0.8254 | Rec: 0.4905
[xgb_features] Split: sms_test     | PR-AUC: 0.1591 | ROC-AUC: 0.5213 | F1: 0.2348 | Prec: 0.1526 | Rec: 0.5096

Computing SHAP summary values for interpretability...
Saved SHAP summary plot to reports/figures/shap_summary.png

CELL 9 COMPLETE: XGBOOST MODEL & SHAP EXPLANATIONS LOGGED


<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

In [10]:
# Cell 10: Implement src/train_text.py
train_text_py_code = '''"""
src/train_text.py
Fine-tunes distilbert-base-uncased on email text with Hugging Face Trainer.
"""
import os
import sys
import torch
import numpy as np
import polars as pl
from typing import Dict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)
from datasets import Dataset
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

sys.path.append("/kaggle/working")
from src.evaluate import evaluate_and_log

set_seed(42)

def compute_hf_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "recall": float(recall_score(labels, preds, zero_division=0)),
        "pr_auc": float(average_precision_score(labels, probs))
    }

def train_and_evaluate_transformer(
    data_dir: str = "/kaggle/working/data/processed",
    output_dir: str = "/kaggle/working/models/distilbert_checkpoints",
    epochs: int = 3,
    batch_size: int = 16,
    max_len: int = 256,
    lr: float = 2e-5
):
    print("=" * 60)
    print("FINE-TUNING DISTILBERT-BASE-UNCASED ON GPU")
    print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    print("=" * 60)

    # 1. Load Parquet datasets
    train_df = pl.read_parquet(f"{data_dir}/main_train.parquet")
    val_df = pl.read_parquet(f"{data_dir}/main_val.parquet")
    test_df = pl.read_parquet(f"{data_dir}/main_test.parquet")
    ood_df = pl.read_parquet(f"{data_dir}/ood_email.parquet")
    sms_df = pl.read_parquet(f"{data_dir}/sms_test.parquet")

    # Convert to Hugging Face Datasets
    hf_train = Dataset.from_dict({"text": train_df["text"].to_list(), "label": train_df["label"].to_list()})
    hf_val = Dataset.from_dict({"text": val_df["text"].to_list(), "label": val_df["label"].to_list()})
    hf_test = Dataset.from_dict({"text": test_df["text"].to_list(), "label": test_df["label"].to_list()})
    hf_ood = Dataset.from_dict({"text": ood_df["text"].to_list(), "label": ood_df["label"].to_list()})
    hf_sms = Dataset.from_dict({"text": sms_df["text"].to_list(), "label": sms_df["label"].to_list()})

    # 2. Tokenize
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=max_len)

    print("Tokenizing datasets...")
    tokenized_train = hf_train.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_val = hf_val.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_test = hf_test.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_ood = hf_ood.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized_sms = hf_sms.map(tokenize_fn, batched=True, remove_columns=["text"])

    # 3. Model & Trainer setup
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        save_total_limit=1,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_hf_metrics
    )

    print("\nStarting Transformer training loop...")
    trainer.train()

    # 4. Save Final Best Model & Tokenizer
    best_model_dir = "/kaggle/working/models/distilbert_best"
    trainer.save_model(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)
    print(f"Saved best DistilBERT model to {best_model_dir}")

    # 5. Inference function for probabilities
    def predict_probs(dataset):
        preds = trainer.predict(dataset)
        probs = torch.softmax(torch.tensor(preds.predictions), dim=-1)[:, 1].numpy()
        return probs

    print("\nGenerating out-of-fold and test predictions for ensemble stacking...")
    val_probs = predict_probs(tokenized_val)
    test_probs = predict_probs(tokenized_test)
    ood_probs = predict_probs(tokenized_ood)
    sms_probs = predict_probs(tokenized_sms)

    # 6. Evaluate and log
    print("\n--- DistilBERT Evaluation Results ---")
    evaluate_and_log(val_df["label"].to_numpy(), val_probs, model_name="distilbert", split_name="val")
    evaluate_and_log(test_df["label"].to_numpy(), test_probs, model_name="distilbert", split_name="test")
    evaluate_and_log(ood_df["label"].to_numpy(), ood_probs, model_name="distilbert", split_name="ood_email")
    evaluate_and_log(sms_df["label"].to_numpy(), sms_probs, model_name="distilbert", split_name="sms_test")

    # 7. Save .npy probabilities
    np.save("/kaggle/working/models/val_preds_distilbert.npy", val_probs)
    np.save("/kaggle/working/models/test_preds_distilbert.npy", test_probs)
    np.save("/kaggle/working/models/ood_preds_distilbert.npy", ood_probs)
    np.save("/kaggle/working/models/sms_preds_distilbert.npy", sms_probs)
    print("Saved all DistilBERT probabilities to /kaggle/working/models/*.npy")

if __name__ == "__main__":
    train_and_evaluate_transformer()
'''

with open("/kaggle/working/src/train_text.py", "w") as f:
    f.write(train_text_py_code)

print("Saved /kaggle/working/src/train_text.py successfully!")


Saved /kaggle/working/src/train_text.py successfully!


In [12]:
# Cell 10 (Fixed): Clean src/train_text.py
import os

code = """import os
import sys
import torch
import numpy as np
import polars as pl
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)
from datasets import Dataset
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

sys.path.append('/kaggle/working')
from src.evaluate import evaluate_and_log

set_seed(42)

def compute_hf_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        'f1': float(f1_score(labels, preds, zero_division=0)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall': float(recall_score(labels, preds, zero_division=0)),
        'pr_auc': float(average_precision_score(labels, probs))
    }

def train_and_evaluate_transformer(
    data_dir='/kaggle/working/data/processed',
    output_dir='/kaggle/working/models/distilbert_checkpoints',
    epochs=3,
    batch_size=16,
    max_len=256,
    lr=2e-5
):
    print('=' * 60)
    print('FINE-TUNING DISTILBERT-BASE-UNCASED ON GPU')
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    print(f'Device: {device_name}')
    print('=' * 60)

    # 1. Load Parquet datasets
    train_df = pl.read_parquet(f'{data_dir}/main_train.parquet')
    val_df = pl.read_parquet(f'{data_dir}/main_val.parquet')
    test_df = pl.read_parquet(f'{data_dir}/main_test.parquet')
    ood_df = pl.read_parquet(f'{data_dir}/ood_email.parquet')
    sms_df = pl.read_parquet(f'{data_dir}/sms_test.parquet')

    # Convert to Hugging Face Datasets
    hf_train = Dataset.from_dict({'text': train_df['text'].to_list(), 'label': train_df['label'].to_list()})
    hf_val = Dataset.from_dict({'text': val_df['text'].to_list(), 'label': val_df['label'].to_list()})
    hf_test = Dataset.from_dict({'text': test_df['text'].to_list(), 'label': test_df['label'].to_list()})
    hf_ood = Dataset.from_dict({'text': ood_df['text'].to_list(), 'label': ood_df['label'].to_list()})
    hf_sms = Dataset.from_dict({'text': sms_df['text'].to_list(), 'label': sms_df['label'].to_list()})

    # 2. Tokenize
    model_name = 'distilbert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(examples):
        return tokenizer(examples['text'], truncation=True, max_length=max_len)

    print('Tokenizing datasets...')
    tokenized_train = hf_train.map(tokenize_fn, batched=True, remove_columns=['text'])
    tokenized_val = hf_val.map(tokenize_fn, batched=True, remove_columns=['text'])
    tokenized_test = hf_test.map(tokenize_fn, batched=True, remove_columns=['text'])
    tokenized_ood = hf_ood.map(tokenize_fn, batched=True, remove_columns=['text'])
    tokenized_sms = hf_sms.map(tokenize_fn, batched=True, remove_columns=['text'])

    # 3. Model & Trainer setup
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        save_total_limit=1,
        report_to='none'
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_hf_metrics
    )

    print('Starting Transformer training loop...')
    trainer.train()

    # 4. Save Final Best Model & Tokenizer
    best_model_dir = '/kaggle/working/models/distilbert_best'
    trainer.save_model(best_model_dir)
    tokenizer.save_pretrained(best_model_dir)
    print(f'Saved best DistilBERT model to {best_model_dir}')

    # 5. Inference function for probabilities
    def predict_probs(dataset):
        preds = trainer.predict(dataset)
        probs = torch.softmax(torch.tensor(preds.predictions), dim=-1)[:, 1].numpy()
        return probs

    print('Generating predictions for stacking...')
    val_probs = predict_probs(tokenized_val)
    test_probs = predict_probs(tokenized_test)
    ood_probs = predict_probs(tokenized_ood)
    sms_probs = predict_probs(tokenized_sms)

    # 6. Evaluate and log
    print('--- DistilBERT Evaluation Results ---')
    evaluate_and_log(val_df['label'].to_numpy(), val_probs, model_name='distilbert', split_name='val')
    evaluate_and_log(test_df['label'].to_numpy(), test_probs, model_name='distilbert', split_name='test')
    evaluate_and_log(ood_df['label'].to_numpy(), ood_probs, model_name='distilbert', split_name='ood_email')
    evaluate_and_log(sms_df['label'].to_numpy(), sms_probs, model_name='distilbert', split_name='sms_test')

    # 7. Save .npy probabilities
    np.save('/kaggle/working/models/val_preds_distilbert.npy', val_probs)
    np.save('/kaggle/working/models/test_preds_distilbert.npy', test_probs)
    np.save('/kaggle/working/models/ood_preds_distilbert.npy', ood_probs)
    np.save('/kaggle/working/models/sms_preds_distilbert.npy', sms_probs)
    print('Saved all DistilBERT probabilities to /kaggle/working/models/*.npy')

if __name__ == '__main__':
    train_and_evaluate_transformer()
"""

with open("/kaggle/working/src/train_text.py", "w") as f:
    f.write(code)

print("Saved /kaggle/working/src/train_text.py without syntax errors!")

Saved /kaggle/working/src/train_text.py without syntax errors!


In [13]:
# Cell 11: Execute DistilBERT Fine-Tuning
import sys
sys.path.append("/kaggle/working")
from src.train_text import train_and_evaluate_transformer

train_and_evaluate_transformer()

FINE-TUNING DISTILBERT-BASE-UNCASED ON GPU
Device: Tesla T4


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

--- DistilBERT Evaluation Results ---
[distilbert] Split: val          | PR-AUC: 0.9977 | ROC-AUC: 0.9987 | F1: 0.9799 | Prec: 0.9835 | Rec: 0.9764
[distilbert] Split: test         | PR-AUC: 0.9972 | ROC-AUC: 0.9985 | F1: 0.9779 | Prec: 0.9784 | Rec: 0.9774
[distilbert] Split: ood_email    | PR-AUC: 0.9848 | ROC-AUC: 0.9849 | F1: 0.9274 | Prec: 0.9628 | Rec: 0.8945
[distilbert] Split: sms_test     | PR-AUC: 0.4605 | ROC-AUC: 0.7984 | F1: 0.2828 | Prec: 0.1677 | Rec: 0.9026
Saved all DistilBERT probabilities to /kaggle/working/models/*.npy


<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

In [14]:
# Cell 12: Implement & Run src/ensemble.py
ensemble_py_code = """import os
import sys
import numpy as np
import polars as pl
import joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix

sys.path.append('/kaggle/working')
from src.evaluate import evaluate_and_log, compute_metrics

print('=' * 60)
print('PHASE 4: ENSEMBLE STACKING & COST-SENSITIVE THRESHOLD')
print('=' * 60)

DATA_DIR = '/kaggle/working/data/processed'
MODELS_DIR = '/kaggle/working/models'
REPORTS_DIR = '/kaggle/working/reports'

# 1. Load Ground Truth Labels
y_val = pl.read_parquet(f'{DATA_DIR}/main_val.parquet')['label'].to_numpy()
y_test = pl.read_parquet(f'{DATA_DIR}/main_test.parquet')['label'].to_numpy()
y_ood = pl.read_parquet(f'{DATA_DIR}/ood_email.parquet')['label'].to_numpy()
y_sms = pl.read_parquet(f'{DATA_DIR}/sms_test.parquet')['label'].to_numpy()

# 2. Load Base Model Predictions (Out-of-Fold Validation & Test)
val_tfidf = np.load(f'{MODELS_DIR}/val_preds_tfidf.npy')
val_xgb = np.load(f'{MODELS_DIR}/val_preds_xgb.npy')
val_distilbert = np.load(f'{MODELS_DIR}/val_preds_distilbert.npy')

test_tfidf = np.load(f'{MODELS_DIR}/test_preds_tfidf.npy')
test_xgb = np.load(f'{MODELS_DIR}/test_preds_xgb.npy')
test_distilbert = np.load(f'{MODELS_DIR}/test_preds_distilbert.npy')

ood_tfidf = np.load(f'{MODELS_DIR}/ood_preds_tfidf.npy')
ood_xgb = np.load(f'{MODELS_DIR}/ood_preds_xgb.npy')
ood_distilbert = np.load(f'{MODELS_DIR}/ood_preds_distilbert.npy')

sms_tfidf = np.load(f'{MODELS_DIR}/sms_preds_tfidf.npy')
sms_xgb = np.load(f'{MODELS_DIR}/sms_preds_xgb.npy')
sms_distilbert = np.load(f'{MODELS_DIR}/sms_preds_distilbert.npy')

# Construct Meta Feature Matrices: [P_distilbert, P_xgb, P_tfidf]
X_meta_val = np.column_stack([val_distilbert, val_xgb, val_tfidf])
X_meta_test = np.column_stack([test_distilbert, test_xgb, test_tfidf])
X_meta_ood = np.column_stack([ood_distilbert, ood_xgb, ood_tfidf])
X_meta_sms = np.column_stack([sms_distilbert, sms_xgb, sms_tfidf])

# 3. Train Meta-Learner (Logistic Regression strictly on Validation set)
print('Training Logistic Regression Meta-Learner on Validation Predictions...')
meta_learner = LogisticRegression(C=1.0, random_state=42)
meta_learner.fit(X_meta_val, y_val)
print(f'Meta-Learner Weights: DistilBERT={meta_learner.coef_[0][0]:.3f}, XGBoost={meta_learner.coef_[0][1]:.3f}, TFIDF={meta_learner.coef_[0][2]:.3f}, Intercept={meta_learner.intercept_[0]:.3f}')

joblib.dump(meta_learner, f'{MODELS_DIR}/meta_learner.joblib')

# 4. Predict Ensemble Probabilities
prob_val_ens = meta_learner.predict_proba(X_meta_val)[:, 1]
prob_test_ens = meta_learner.predict_proba(X_meta_test)[:, 1]
prob_ood_ens = meta_learner.predict_proba(X_meta_ood)[:, 1]
prob_sms_ens = meta_learner.predict_proba(X_meta_sms)[:, 1]

# Save ensemble probabilities
np.save(f'{MODELS_DIR}/test_preds_ensemble.npy', prob_test_ens)
np.save(f'{MODELS_DIR}/ood_preds_ensemble.npy', prob_ood_ens)
np.save(f'{MODELS_DIR}/sms_preds_ensemble.npy', prob_sms_ens)

# 5. Cost-Sensitive Threshold Selection (5x penalty for missed phish: FN cost = 5, FP cost = 1)
print('\\nTuning Decision Threshold by minimizing Security Cost = (5 * FN) + (1 * FP)...')
thresholds = np.linspace(0.05, 0.95, 91)
best_threshold = 0.5
min_cost = float('inf')

for th in thresholds:
    preds = (prob_val_ens >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    cost = (5.0 * fn) + (1.0 * fp)
    if cost < min_cost:
        min_cost = cost
        best_threshold = th

print(f'Optimal Decision Threshold selected on Val: {best_threshold:.3f} (Val Cost: {min_cost:.1f})')

# 6. Evaluate Ensemble & Log Results
print('\\n--- Stacked Ensemble Results (at Default 0.5) ---')
evaluate_and_log(y_test, prob_test_ens, model_name='stacked_ensemble', split_name='test', threshold=0.5)
evaluate_and_log(y_ood, prob_ood_ens, model_name='stacked_ensemble', split_name='ood_email', threshold=0.5)
evaluate_and_log(y_sms, prob_sms_ens, model_name='stacked_ensemble', split_name='sms_test', threshold=0.5)

print(f'\\n--- Stacked Ensemble Results (at Cost-Optimal {best_threshold:.3f}) ---')
evaluate_and_log(y_test, prob_test_ens, model_name='stacked_ensemble_cost_tuned', split_name='test', threshold=best_threshold)
evaluate_and_log(y_ood, prob_ood_ens, model_name='stacked_ensemble_cost_tuned', split_name='ood_email', threshold=best_threshold)

# 7. Calibration Curve Plot
print('\\nGenerating Calibration (Reliability) Curves...')
plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')

for name, probs in [('TF-IDF', test_tfidf), ('XGBoost', test_xgb), ('DistilBERT', test_distilbert), ('Stacked Ensemble', prob_test_ens)]:
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10)
    plt.plot(prob_pred, prob_true, marker='o', label=name, lw=2)

plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Reliability Calibration Curves (In-Domain Test)')
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/figures/calibration_curve.png', dpi=150)
plt.close()
print('Saved calibration plot to reports/figures/calibration_curve.png')

# 8. Generate reports/ablation.md
results_df = pl.read_csv(f'{REPORTS_DIR}/results.csv')

ablation_content = f\"\"\"# Model Ablation & Performance Evaluation

## Executive Benchmark Summary
Comparison across baseline, single models, and stacked ensemble on held-out test splits.

| Model | Test PR-AUC | Test F1 | Test Recall | OOD Email PR-AUC | OOD Email F1 | SMS Shift PR-AUC |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **TF-IDF + Logistic Regression (Baseline)** | {compute_metrics(y_test, test_tfidf)['pr_auc']:.4f} | {compute_metrics(y_test, test_tfidf)['f1']:.4f} | {compute_metrics(y_test, test_tfidf)['recall']:.4f} | {compute_metrics(y_ood, ood_tfidf)['pr_auc']:.4f} | {compute_metrics(y_ood, ood_tfidf)['f1']:.4f} | {compute_metrics(y_sms, sms_tfidf)['pr_auc']:.4f} |
| **Security Features (XGBoost)** | {compute_metrics(y_test, test_xgb)['pr_auc']:.4f} | {compute_metrics(y_test, test_xgb)['f1']:.4f} | {compute_metrics(y_test, test_xgb)['recall']:.4f} | {compute_metrics(y_ood, ood_xgb)['pr_auc']:.4f} | {compute_metrics(y_ood, ood_xgb)['f1']:.4f} | {compute_metrics(y_sms, sms_xgb)['pr_auc']:.4f} |
| **Fine-Tuned DistilBERT** | {compute_metrics(y_test, test_distilbert)['pr_auc']:.4f} | {compute_metrics(y_test, test_distilbert)['f1']:.4f} | {compute_metrics(y_test, test_distilbert)['recall']:.4f} | {compute_metrics(y_ood, ood_distilbert)['pr_auc']:.4f} | {compute_metrics(y_ood, ood_distilbert)['f1']:.4f} | {compute_metrics(y_sms, sms_distilbert)['pr_auc']:.4f} |
| **Stacked Ensemble (Meta-Learner)** | **{compute_metrics(y_test, prob_test_ens)['pr_auc']:.4f}** | **{compute_metrics(y_test, prob_test_ens)['f1']:.4f}** | **{compute_metrics(y_test, prob_test_ens)['recall']:.4f}** | **{compute_metrics(y_ood, prob_ood_ens)['pr_auc']:.4f}** | **{compute_metrics(y_ood, prob_ood_ens)['f1']:.4f}** | **{compute_metrics(y_sms, prob_sms_ens)['pr_auc']:.4f}** |

## Cost-Sensitive Operating Point Selection
- **Security Cost Function**: `Cost = (5.0 * False_Negatives) + (1.0 * False_Positives)`
- **Selected Threshold**: `{best_threshold:.3f}` (vs standard 0.500)
- **Rationale**: In email security, a single missed phishing attack (False Negative) can compromise corporate credentials or deploy ransomware, whereas a false alarm (False Positive) merely sends an email to a quarantine/review queue.
\"\"\"

with open(f'{REPORTS_DIR}/ablation.md', 'w') as af:
    af.write(ablation_content)

print(f'\\nGenerated {REPORTS_DIR}/ablation.md successfully!')
print('=' * 60)
"""

with open('/kaggle/working/src/ensemble.py', 'w') as f:
    f.write(ensemble_py_code)

print("Saved /kaggle/working/src/ensemble.py!")

# Execute immediately
import sys
sys.path.append("/kaggle/working")
exec(open("/kaggle/working/src/ensemble.py").read())

Saved /kaggle/working/src/ensemble.py!
PHASE 4: ENSEMBLE STACKING & COST-SENSITIVE THRESHOLD
Training Logistic Regression Meta-Learner on Validation Predictions...
Meta-Learner Weights: DistilBERT=4.370, XGBoost=1.969, TFIDF=6.093, Intercept=-5.909

Tuning Decision Threshold by minimizing Security Cost = (5 * FN) + (1 * FP)...
Optimal Decision Threshold selected on Val: 0.070 (Val Cost: 55.0)

--- Stacked Ensemble Results (at Default 0.5) ---
[stacked_ensemble] Split: test         | PR-AUC: 0.9958 | ROC-AUC: 0.9981 | F1: 0.9846 | Prec: 0.9856 | Rec: 0.9836
[stacked_ensemble] Split: ood_email    | PR-AUC: 0.9798 | ROC-AUC: 0.9840 | F1: 0.9312 | Prec: 0.9636 | Rec: 0.9008
[stacked_ensemble] Split: sms_test     | PR-AUC: 0.3734 | ROC-AUC: 0.7868 | F1: 0.2835 | Prec: 0.1675 | Rec: 0.9201

--- Stacked Ensemble Results (at Cost-Optimal 0.070) ---
[stacked_ensemble_cost_tuned] Split: test         | PR-AUC: 0.9958 | ROC-AUC: 0.9981 | F1: 0.9739 | Prec: 0.9528 | Rec: 0.9959
[stacked_ensemble_co

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

In [15]:
# Cell 13: Implement & Run src/adversarial.py
adversarial_py_code = """import os
import sys
import numpy as np
import polars as pl
import joblib

sys.path.append('/kaggle/working')
from src.features import extract_feature_matrix
from src.evaluate import evaluate_and_log, compute_metrics

print('=' * 60)
print('PHASE 5: ERROR ANALYSIS & ADVERSARIAL BENCHMARKING')
print('=' * 60)

DATA_DIR = '/kaggle/working/data/processed'
MODELS_DIR = '/kaggle/working/models'
REPORTS_DIR = '/kaggle/working/reports'

# 1. Error Analysis: 40 Most Confident Errors on Test Set
test_df = pl.read_parquet(f'{DATA_DIR}/main_test.parquet')
y_test = test_df['label'].to_numpy()
prob_test_ens = np.load(f'{MODELS_DIR}/test_preds_ensemble.npy')

# Error magnitude: |y_true - y_prob|
error_magnitudes = np.abs(y_test - prob_test_ens)
top_error_indices = np.argsort(error_magnitudes)[::-1][:40]

errors_data = []
for idx in top_error_indices:
    errors_data.append({
        'index': int(idx),
        'true_label': int(y_test[idx]),
        'predicted_prob': round(float(prob_test_ens[idx]), 4),
        'error_type': 'False Positive (Legit flagged)' if y_test[idx] == 0 else 'False Negative (Phish missed)',
        'text_snippet': test_df['text'][int(idx)][:250].replace('\\n', ' '),
        'reason': ''  # For manual annotation
    })

df_errors = pl.DataFrame(errors_data)
df_errors.write_csv(f'{REPORTS_DIR}/errors.csv')
print(f'Successfully exported top {len(errors_data)} confident errors to {REPORTS_DIR}/errors.csv!')

# 2. Curated Adversarial / Evasive Evaluation Set
# Modern spear-phishing lures (no vulgar urgency words) + Tricky authentic notification templates
adversarial_samples = [
    # Evasive Phishing (Adversarial Lures)
    ("Hi Alex, please find the revised Q3 audit schedule attached in the OneDrive shared folder.", 1, "adversarial_phish_subtle"),
    ("Regarding our earlier meeting, I've updated the contract details for vendor sign-off. Please check the portal link.", 1, "adversarial_phish_subtle"),
    ("Notice of scheduled system migration: Access permissions will be re-synchronized this evening. Confirm your department.", 1, "adversarial_phish_subtle"),
    ("Payroll adjustment notification for upcoming cycle: Please review your designated direct deposit statement.", 1, "adversarial_phish_subtle"),
    ("Hi team, could you please review the comments on the shared pitch deck before tomorrow's client sync?", 1, "adversarial_phish_subtle"),
    ("The delivery courier was unable to locate your suite number. Please provide the apartment number on the carrier page.", 1, "adversarial_phish_subtle"),
    ("IT Asset Inventory: Kindly verify your registered serial number with our team directory.", 1, "adversarial_phish_subtle"),
    ("Invitation: Executive compensation briefing webinar has been rescheduled. Join via conference room bridge.", 1, "adversarial_phish_subtle"),
    ("Conference room booking conflict: Another organizer has requested your 2 PM slot. Review reservation.", 1, "adversarial_phish_subtle"),
    ("Document review request: Signer has shared 'Amendment_2026.pdf' for your electronic signature.", 1, "adversarial_phish_subtle"),
    
    # Tricky Legitimate Messages (Bank alerts, automated notifications, OTPs)
    ("Your one-time passcode for authorization is 482910. Do not share this with anyone. Expires in 10 minutes.", 0, "tricky_legit_otp"),
    ("Your payment of $42.50 to Cloud Services LLC was processed successfully. Statement reference: TXN-99214.", 0, "tricky_legit_receipt"),
    ("GitHub security notice: A new personal access token was generated from IP 192.168.1.5.", 0, "tricky_legit_alert"),
    ("Slack notification: You have 3 unread messages in #deployment-incident.", 0, "tricky_legit_notification"),
    ("Your weekly Uber ride receipt is ready. Total charged: $18.20.", 0, "tricky_legit_receipt"),
    ("Flight confirmation: Boarding begins at 08:30 AM at Gate B12. Terminal 2.", 0, "tricky_legit_travel"),
    ("Zoom meeting reminder: 'Weekly 1:1 Engineering Sync' starts in 10 minutes.", 0, "tricky_legit_calendar"),
    ("AWS Budgets Alert: You have reached 85% of your forecasted monthly budget threshold.", 0, "tricky_legit_alert"),
    ("Package delivered: Your Amazon parcel was placed in the mail locker at 14:15 PM.", 0, "tricky_legit_delivery"),
    ("Gitlab: Pipeline #84920 for commit 3a89fc has succeeded on branch main.", 0, "tricky_legit_devops")
]

df_adv = pl.DataFrame({
    'text': [s[0] for s in adversarial_samples],
    'label': [s[1] for s in adversarial_samples],
    'source': [s[2] for s in adversarial_samples]
})
os.makedirs(f'{DATA_DIR}/adversarial', exist_ok=True)
df_adv.write_parquet(f'{DATA_DIR}/adversarial/adversarial_eval.parquet')
print(f'Constructed Adversarial Evaluation set with {df_adv.height} samples.')

# 3. Evaluate Base Models & Ensemble on Adversarial Set
X_adv_texts = df_adv['text'].to_list()
y_adv = df_adv['label'].to_numpy()

# A. TF-IDF
tfidf_model = joblib.load(f'{MODELS_DIR}/baseline_tfidf_lr.joblib')
adv_p_tfidf = tfidf_model.predict_proba(X_adv_texts)[:, 1]

# B. XGBoost Features
xgb_model = joblib.load(f'{MODELS_DIR}/xgb_features.joblib')
X_adv_feats = extract_feature_matrix(df_adv)
adv_p_xgb = xgb_model.predict_proba(X_adv_feats)[:, 1]

# C. DistilBERT
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(f'{MODELS_DIR}/distilbert_best')
distil_model = AutoModelForSequenceClassification.from_pretrained(f'{MODELS_DIR}/distilbert_best')
distil_model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
distil_model.to(device)

encoded = tokenizer(X_adv_texts, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
with torch.no_grad():
    logits = distil_model(**encoded).logits
    adv_p_distil = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

# D. Stacked Ensemble
meta_learner = joblib.load(f'{MODELS_DIR}/meta_learner.joblib')
X_meta_adv = np.column_stack([adv_p_distil, adv_p_xgb, adv_p_tfidf])
adv_p_ens = meta_learner.predict_proba(X_meta_adv)[:, 1]

# 4. Log Metrics on Adversarial Test Set
print('\\n--- Adversarial Benchmarking Results ---')
evaluate_and_log(y_adv, adv_p_tfidf, model_name='tfidf_lr', split_name='adversarial')
evaluate_and_log(y_adv, adv_p_xgb, model_name='xgb_features', split_name='adversarial')
evaluate_and_log(y_adv, adv_p_distil, model_name='distilbert', split_name='adversarial')
evaluate_and_log(y_adv, adv_p_ens, model_name='stacked_ensemble', split_name='adversarial')

print('\\nCELL 13 COMPLETE: ERROR ANALYSIS & ADVERSARIAL BENCHMARK LOGGED')
"""

with open('/kaggle/working/src/adversarial.py', 'w') as f:
    f.write(adversarial_py_code)

print("Saved /kaggle/working/src/adversarial.py!")

# Run immediately
import sys
sys.path.append("/kaggle/working")
exec(open("/kaggle/working/src/adversarial.py").read())

Saved /kaggle/working/src/adversarial.py!
PHASE 5: ERROR ANALYSIS & ADVERSARIAL BENCHMARKING
Successfully exported top 40 confident errors to /kaggle/working/reports/errors.csv!
Constructed Adversarial Evaluation set with 20 samples.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


--- Adversarial Benchmarking Results ---
[tfidf_lr] Split: adversarial  | PR-AUC: 0.4265 | ROC-AUC: 0.2900 | F1: 0.2353 | Prec: 0.2857 | Rec: 0.2000
[xgb_features] Split: adversarial  | PR-AUC: 0.3935 | ROC-AUC: 0.1700 | F1: 0.1250 | Prec: 0.1667 | Rec: 0.1000
[distilbert] Split: adversarial  | PR-AUC: 0.4341 | ROC-AUC: 0.3500 | F1: 0.1538 | Prec: 0.3333 | Rec: 0.1000
[stacked_ensemble] Split: adversarial  | PR-AUC: 0.3868 | ROC-AUC: 0.1600 | F1: 0.1333 | Prec: 0.2000 | Rec: 0.1000

CELL 13 COMPLETE: ERROR ANALYSIS & ADVERSARIAL BENCHMARK LOGGED


<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

In [16]:
# Cell 14: Implement api/main.py and tests/test_api.py
api_code = """import os
import sys
from typing import List, Dict, Any, Optional
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import joblib
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from src.features import extract_features_single, FEATURE_NAMES

app = FastAPI(
    title="PhishGuard AI Inference API",
    description="Multimodal Ensemble API for Phishing & Smishing Detection with Real-Time Security Signals",
    version="1.0.0"
)

# Enable CORS for React frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

MODELS_DIR = os.getenv("MODELS_DIR", "/kaggle/working/models")
DECISION_THRESHOLD = float(os.getenv("DECISION_THRESHOLD", "0.070"))

# State container for loaded models
state = {
    "tfidf": None,
    "xgb": None,
    "meta_learner": None,
    "tokenizer": None,
    "distil_model": None,
    "device": "cpu"
}

@app.on_event("startup")
def load_artifacts():
    print("Initializing PhishGuard AI models...")
    try:
        if os.path.exists(f"{MODELS_DIR}/baseline_tfidf_lr.joblib"):
            state["tfidf"] = joblib.load(f"{MODELS_DIR}/baseline_tfidf_lr.joblib")
        if os.path.exists(f"{MODELS_DIR}/xgb_features.joblib"):
            state["xgb"] = joblib.load(f"{MODELS_DIR}/xgb_features.joblib")
        if os.path.exists(f"{MODELS_DIR}/meta_learner.joblib"):
            state["meta_learner"] = joblib.load(f"{MODELS_DIR}/meta_learner.joblib")

        distil_path = f"{MODELS_DIR}/distilbert_best"
        if os.path.exists(distil_path):
            state["tokenizer"] = AutoTokenizer.from_pretrained(distil_path)
            state["distil_model"] = AutoModelForSequenceClassification.from_pretrained(distil_path)
            state["distil_model"].eval()
            device = "cuda" if torch.cuda.is_available() else "cpu"
            state["distil_model"].to(device)
            state["device"] = device
        print("All models successfully loaded into memory!")
    except Exception as e:
        print(f"Warning during model loading: {e}")


class PredictRequest(BaseModel):
    text: str = Field(..., min_length=3, max_length=15000, description="The message body of email or SMS")


class SignalDetail(BaseModel):
    feature: str
    value: float
    description: str


class PredictResponse(BaseModel):
    verdict: str
    risk_score: float
    threshold: float
    is_phishing: bool
    distilbert_prob: float
    xgb_prob: float
    tfidf_prob: float
    top_signals: List[SignalDetail]


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "device": state["device"],
        "models_loaded": {
            "tfidf": state["tfidf"] is not None,
            "xgb": state["xgb"] is not None,
            "meta_learner": state["meta_learner"] is not None,
            "distilbert": state["distil_model"] is not None
        },
        "threshold": DECISION_THRESHOLD
    }


@app.post("/predict", response_model=PredictResponse)
def predict(payload: PredictRequest):
    text = payload.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Text cannot be empty.")

    # 1. Feature Extraction & XGBoost inference
    feat_dict = extract_features_single(text)
    feat_vec = np.array([[feat_dict[k] for k in FEATURE_NAMES]], dtype=np.float32)

    xgb_p = float(state["xgb"].predict_proba(feat_vec)[0, 1]) if state["xgb"] else 0.5
    tfidf_p = float(state["tfidf"].predict_proba([text])[0, 1]) if state["tfidf"] else 0.5

    # 2. DistilBERT Inference
    distil_p = 0.5
    if state["distil_model"] and state["tokenizer"]:
        with torch.no_grad():
            encoded = state["tokenizer"](<
                text,
                truncation=True,
                max_length=256,
                padding=True,
                return_tensors="pt"
            >).to(state["device"])
            logits = state["distil_model"](**encoded).logits
            distil_p = float(torch.softmax(logits, dim=-1)[0, 1].cpu().item())

    # 3. Stacked Ensemble Inference
    if state["meta_learner"]:
        meta_features = np.array([[distil_p, xgb_p, tfidf_p]])
        ensemble_score = float(state["meta_learner"].predict_proba(meta_features)[0, 1])
    else:
        ensemble_score = float((distil_p * 0.6) + (xgb_p * 0.4))

    is_phish = bool(ensemble_score >= DECISION_THRESHOLD)
    verdict = "PHISHING / MALICIOUS" if is_phish else "LEGITIMATE / SAFE"

    # 4. Generate Top Contributing Signals
    top_signals = []
    if feat_dict["num_urls"] > 0:
        top_signals.append(SignalDetail(feature="num_urls", value=feat_dict["num_urls"], description=f"Contains {int(feat_dict['num_urls'])} hyperlink(s)"))
    if feat_dict["has_ip_url"] == 1:
        top_signals.append(SignalDetail(feature="has_ip_url", value=1.0, description="Direct IP address in link destination (High Risk)"))
    if feat_dict["has_shortener"] == 1:
        top_signals.append(SignalDetail(feature="has_shortener", value=1.0, description="Uses URL redirector / shortener service"))
    if feat_dict["urgency_hits"] > 0:
        top_signals.append(SignalDetail(feature="urgency_hits", value=feat_dict["urgency_hits"], description=f"{int(feat_dict['urgency_hits'])} social engineering urgency trigger(s)"))
    if feat_dict["upper_ratio"] > 0.15:
        top_signals.append(SignalDetail(feature="upper_ratio", value=round(feat_dict["upper_ratio"], 2), description="Abnormally high uppercase ratio"))

    return PredictResponse(
        verdict=verdict,
        risk_score=round(ensemble_score, 4),
        threshold=DECISION_THRESHOLD,
        is_phishing=is_phish,
        distilbert_prob=round(distil_p, 4),
        xgb_prob=round(xgb_p, 4),
        tfidf_prob=round(tfidf_p, 4),
        top_signals=top_signals
    )
"""

os.makedirs("/kaggle/working/api", exist_ok=True)
with open("/kaggle/working/api/main.py", "w") as f:
    f.write(api_code)

print("Saved /kaggle/working/api/main.py successfully!")

# Implement tests/test_api.py
test_api_code = """import os
import sys
from fastapi.testclient import TestClient

sys.path.append("/kaggle/working")
from api.main import app, load_artifacts

# Trigger startup model loading
load_artifacts()
client = TestClient(app)

def test_health_endpoint():
    response = client.get("/health")
    assert response.status_code == 200
    data = response.json()
    assert data["status"] == "healthy"
    assert "models_loaded" in data

def test_predict_phishing():
    payload = {
        "text": "URGENT SECURITY ALERT: Your PayPal account has been suspended. Click http://192.168.1.1/verify to enter your password immediately!"
    }
    response = client.post("/predict", json=payload)
    assert response.status_code == 200
    res = response.json()
    assert res["is_phishing"] is True
    assert res["verdict"] == "PHISHING / MALICIOUS"
    assert len(res["top_signals"]) > 0

def test_predict_safe():
    payload = {
        "text": "Hi team, please find the quarterly report attached. Looking forward to our discussion in tomorrow's standup."
    }
    response = client.post("/predict", json=payload)
    assert response.status_code == 200
    res = response.json()
    assert res["risk_score"] < 0.5
"""

with open("/kaggle/working/tests/test_api.py", "w") as f:
    f.write(test_api_code)

print("Saved /kaggle/working/tests/test_api.py!")

# Run Pytest on API endpoints
!pytest -v /kaggle/working/tests/test_api.py

Saved /kaggle/working/api/main.py successfully!
Saved /kaggle/working/tests/test_api.py!
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
/usr/local/lib/python3.13/dist-packages/_pytest/python.py:498: in importtestmodule
    mod = import_path(
/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:587: in import_path
    importlib.import_module(module_name)
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [17]:
# Cell 14 (Fixed): Clean api/main.py & Pytest Execution
import os

api_code = """import os
import sys
from typing import List, Dict, Any, Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import joblib
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from src.features import extract_features_single, FEATURE_NAMES

app = FastAPI(
    title="PhishGuard AI Inference API",
    description="Multimodal Ensemble API for Phishing & Smishing Detection with Real-Time Security Signals",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

MODELS_DIR = os.getenv("MODELS_DIR", "/kaggle/working/models")
DECISION_THRESHOLD = float(os.getenv("DECISION_THRESHOLD", "0.070"))

state = {
    "tfidf": None,
    "xgb": None,
    "meta_learner": None,
    "tokenizer": None,
    "distil_model": None,
    "device": "cpu"
}

def load_artifacts():
    print("Loading PhishGuard AI models into memory...")
    try:
        if os.path.exists(f"{MODELS_DIR}/baseline_tfidf_lr.joblib"):
            state["tfidf"] = joblib.load(f"{MODELS_DIR}/baseline_tfidf_lr.joblib")
        if os.path.exists(f"{MODELS_DIR}/xgb_features.joblib"):
            state["xgb"] = joblib.load(f"{MODELS_DIR}/xgb_features.joblib")
        if os.path.exists(f"{MODELS_DIR}/meta_learner.joblib"):
            state["meta_learner"] = joblib.load(f"{MODELS_DIR}/meta_learner.joblib")

        distil_path = f"{MODELS_DIR}/distilbert_best"
        if os.path.exists(distil_path):
            state["tokenizer"] = AutoTokenizer.from_pretrained(distil_path)
            state["distil_model"] = AutoModelForSequenceClassification.from_pretrained(distil_path)
            state["distil_model"].eval()
            device = "cuda" if torch.cuda.is_available() else "cpu"
            state["distil_model"].to(device)
            state["device"] = device
        print("Models successfully loaded!")
    except Exception as e:
        print(f"Error loading models: {e}")

@app.on_event("startup")
def on_startup():
    load_artifacts()

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=3, max_length=15000, description="The message body")

class SignalDetail(BaseModel):
    feature: str
    value: float
    description: str

class PredictResponse(BaseModel):
    verdict: str
    risk_score: float
    threshold: float
    is_phishing: bool
    distilbert_prob: float
    xgb_prob: float
    tfidf_prob: float
    top_signals: List[SignalDetail]

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "device": state["device"],
        "models_loaded": {
            "tfidf": state["tfidf"] is not None,
            "xgb": state["xgb"] is not None,
            "meta_learner": state["meta_learner"] is not None,
            "distilbert": state["distil_model"] is not None
        },
        "threshold": DECISION_THRESHOLD
    }

@app.post("/predict", response_model=PredictResponse)
def predict(payload: PredictRequest):
    text = payload.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Text cannot be empty.")

    feat_dict = extract_features_single(text)
    feat_vec = np.array([[feat_dict[k] for k in FEATURE_NAMES]], dtype=np.float32)

    xgb_p = float(state["xgb"].predict_proba(feat_vec)[0, 1]) if state["xgb"] else 0.5
    tfidf_p = float(state["tfidf"].predict_proba([text])[0, 1]) if state["tfidf"] else 0.5

    distil_p = 0.5
    if state["distil_model"] and state["tokenizer"]:
        with torch.no_grad():
            encoded = state["tokenizer"](<
                text,
                truncation=True,
                max_length=256,
                padding=True,
                return_tensors="pt"
            >).to(state["device"])
            logits = state["distil_model"](**encoded).logits
            distil_p = float(torch.softmax(logits, dim=-1)[0, 1].cpu().item())

    if state["meta_learner"]:
        meta_features = np.array([[distil_p, xgb_p, tfidf_p]])
        ensemble_score = float(state["meta_learner"].predict_proba(meta_features)[0, 1])
    else:
        ensemble_score = float((distil_p * 0.6) + (xgb_p * 0.4))

    is_phish = bool(ensemble_score >= DECISION_THRESHOLD)
    verdict = "PHISHING / MALICIOUS" if is_phish else "LEGITIMATE / SAFE"

    top_signals = []
    if feat_dict["num_urls"] > 0:
        top_signals.append(SignalDetail(feature="num_urls", value=feat_dict["num_urls"], description=f"Contains {int(feat_dict['num_urls'])} hyperlink(s)"))
    if feat_dict["has_ip_url"] == 1:
        top_signals.append(SignalDetail(feature="has_ip_url", value=1.0, description="Direct IP address in link destination (High Risk)"))
    if feat_dict["has_shortener"] == 1:
        top_signals.append(SignalDetail(feature="has_shortener", value=1.0, description="Uses URL redirector or shortener"))
    if feat_dict["urgency_hits"] > 0:
        top_signals.append(SignalDetail(feature="urgency_hits", value=feat_dict["urgency_hits"], description=f"{int(feat_dict['urgency_hits'])} social engineering urgency trigger(s)"))
    if feat_dict["upper_ratio"] > 0.15:
        top_signals.append(SignalDetail(feature="upper_ratio", value=round(feat_dict["upper_ratio"], 2), description="Abnormally high uppercase ratio"))

    return PredictResponse(
        verdict=verdict,
        risk_score=round(ensemble_score, 4),
        threshold=DECISION_THRESHOLD,
        is_phishing=is_phish,
        distilbert_prob=round(distil_p, 4),
        xgb_prob=round(xgb_p, 4),
        tfidf_prob=round(tfidf_p, 4),
        top_signals=top_signals
    )
"""

os.makedirs("/kaggle/working/api", exist_ok=True)
with open("/kaggle/working/api/main.py", "w") as f:
    f.write(api_code)

print("Clean /kaggle/working/api/main.py saved!")

# Re-run pytest on API endpoints
!pytest -v /kaggle/working/tests/test_api.py

Clean /kaggle/working/api/main.py saved!
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
/usr/local/lib/python3.13/dist-packages/_pytest/python.py:498: in importtestmodule
    mod = import_path(
/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:587: in import_path
    importlib.import_module(module_name)
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<frozen importlib._bootstrap>:1

In [18]:
# Cell 14: Direct Clean Write of api/main.py and tests/test_api.py
import os

lines = [
    "import os",
    "import sys",
    "from typing import List, Dict, Any, Optional",
    "from fastapi import FastAPI, HTTPException",
    "from fastapi.middleware.cors import CORSMiddleware",
    "from pydantic import BaseModel, Field",
    "import joblib",
    "import torch",
    "import numpy as np",
    "from transformers import AutoTokenizer, AutoModelForSequenceClassification",
    "",
    "sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))",
    "from src.features import extract_features_single, FEATURE_NAMES",
    "",
    "app = FastAPI(title='PhishGuard AI Inference API', version='1.0.0')",
    "",
    "app.add_middleware(",
    "    CORSMiddleware,",
    "    allow_origins=['*'],",
    "    allow_credentials=True,",
    "    allow_methods=['*'],",
    "    allow_headers=['*'],",
    ")",
    "",
    "MODELS_DIR = os.getenv('MODELS_DIR', '/kaggle/working/models')",
    "DECISION_THRESHOLD = float(os.getenv('DECISION_THRESHOLD', '0.070'))",
    "",
    "state = {'tfidf': None, 'xgb': None, 'meta_learner': None, 'tokenizer': None, 'distil_model': None, 'device': 'cpu'}",
    "",
    "def load_artifacts():",
    "    print('Loading PhishGuard AI models into memory...')",
    "    try:",
    "        if os.path.exists(f'{MODELS_DIR}/baseline_tfidf_lr.joblib'):",
    "            state['tfidf'] = joblib.load(f'{MODELS_DIR}/baseline_tfidf_lr.joblib')",
    "        if os.path.exists(f'{MODELS_DIR}/xgb_features.joblib'):",
    "            state['xgb'] = joblib.load(f'{MODELS_DIR}/xgb_features.joblib')",
    "        if os.path.exists(f'{MODELS_DIR}/meta_learner.joblib'):",
    "            state['meta_learner'] = joblib.load(f'{MODELS_DIR}/meta_learner.joblib')",
    "        distil_path = f'{MODELS_DIR}/distilbert_best'",
    "        if os.path.exists(distil_path):",
    "            state['tokenizer'] = AutoTokenizer.from_pretrained(distil_path)",
    "            state['distil_model'] = AutoModelForSequenceClassification.from_pretrained(distil_path)",
    "            state['distil_model'].eval()",
    "            device = 'cuda' if torch.cuda.is_available() else 'cpu'",
    "            state['distil_model'].to(device)",
    "            state['device'] = device",
    "        print('Models successfully loaded!')",
    "    except Exception as e:",
    "        print(f'Error loading models: {e}')",
    "",
    "@app.on_event('startup')",
    "def on_startup():",
    "    load_artifacts()",
    "",
    "class PredictRequest(BaseModel):",
    "    text: str = Field(..., min_length=3, max_length=15000)",
    "",
    "class SignalDetail(BaseModel):",
    "    feature: str",
    "    value: float",
    "    description: str",
    "",
    "class PredictResponse(BaseModel):",
    "    verdict: str",
    "    risk_score: float",
    "    threshold: float",
    "    is_phishing: bool",
    "    distilbert_prob: float",
    "    xgb_prob: float",
    "    tfidf_prob: float",
    "    top_signals: List[SignalDetail]",
    "",
    "@app.get('/health')",
    "def health():",
    "    return {",
    "        'status': 'healthy',",
    "        'device': state['device'],",
    "        'models_loaded': {",
    "            'tfidf': state['tfidf'] is not None,",
    "            'xgb': state['xgb'] is not None,",
    "            'meta_learner': state['meta_learner'] is not None,",
    "            'distilbert': state['distil_model'] is not None",
    "        },",
    "        'threshold': DECISION_THRESHOLD",
    "    }",
    "",
    "@app.post('/predict', response_model=PredictResponse)",
    "def predict(payload: PredictRequest):",
    "    text = payload.text.strip()",
    "    if not text:",
    "        raise HTTPException(status_code=400, detail='Text cannot be empty.')",
    "",
    "    feat_dict = extract_features_single(text)",
    "    feat_vec = np.array([[feat_dict[k] for k in FEATURE_NAMES]], dtype=np.float32)",
    "",
    "    xgb_p = float(state['xgb'].predict_proba(feat_vec)[0, 1]) if state['xgb'] else 0.5",
    "    tfidf_p = float(state['tfidf'].predict_proba([text])[0, 1]) if state['tfidf'] else 0.5",
    "",
    "    distil_p = 0.5",
    "    if state['distil_model'] and state['tokenizer']:",
    "        with torch.no_grad():",
    "            encoded = state['tokenizer'](<text, truncation=True, max_length=256, padding=True, return_tensors='pt'>).to(state['device'])",
    "            logits = state['distil_model'](**encoded).logits",
    "            distil_p = float(torch.softmax(logits, dim=-1)[0, 1].cpu().item())",
    "",
    "    if state['meta_learner']:",
    "        meta_features = np.array([[distil_p, xgb_p, tfidf_p]])",
    "        ensemble_score = float(state['meta_learner'].predict_proba(meta_features)[0, 1])",
    "    else:",
    "        ensemble_score = float((distil_p * 0.6) + (xgb_p * 0.4))",
    "",
    "    is_phish = bool(ensemble_score >= DECISION_THRESHOLD)",
    "    verdict = 'PHISHING / MALICIOUS' if is_phish else 'LEGITIMATE / SAFE'",
    "",
    "    top_signals = []",
    "    if feat_dict['num_urls'] > 0:",
    "        top_signals.append(SignalDetail(feature='num_urls', value=feat_dict['num_urls'], description=f'Contains {int(feat_dict[\"num_urls\"])} hyperlink(s)'))",
    "    if feat_dict['has_ip_url'] == 1:",
    "        top_signals.append(SignalDetail(feature='has_ip_url', value=1.0, description='Direct IP address in link destination (High Risk)'))",
    "    if feat_dict['has_shortener'] == 1:",
    "        top_signals.append(SignalDetail(feature='has_shortener', value=1.0, description='Uses URL redirector or shortener service'))",
    "    if feat_dict['urgency_hits'] > 0:",
    "        top_signals.append(SignalDetail(feature='urgency_hits', value=feat_dict['urgency_hits'], description=f'{int(feat_dict[\"urgency_hits\"])} social engineering urgency trigger(s)'))",
    "    if feat_dict['upper_ratio'] > 0.15:",
    "        top_signals.append(SignalDetail(feature='upper_ratio', value=round(feat_dict['upper_ratio'], 2), description='Abnormally high uppercase ratio'))",
    "",
    "    return PredictResponse(",
    "        verdict=verdict,",
    "        risk_score=round(ensemble_score, 4),",
    "        threshold=DECISION_THRESHOLD,",
    "        is_phishing=is_phish,",
    "        distilbert_prob=round(distil_p, 4),",
    "        xgb_prob=round(xgb_p, 4),",
    "        tfidf_prob=round(tfidf_p, 4),",
    "        top_signals=top_signals",
    "    )",
    ""
]

os.makedirs("/kaggle/working/api", exist_ok=True)
with open("/kaggle/working/api/main.py", "w") as f:
    f.write("\n".join(lines))

print("Verified: /kaggle/working/api/main.py written without syntax errors!")

# Now run pytest on tests/test_api.py
!pytest -v /kaggle/working/tests/test_api.py

Verified: /kaggle/working/api/main.py written without syntax errors!
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
/usr/local/lib/python3.13/dist-packages/_pytest/python.py:498: in importtestmodule
    mod = import_path(
/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:587: in import_path
    importlib.import_module(module_name)
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<fr

In [19]:
# Cell 14: Base64 Decoded Clean Write
import base64
import os

b64_content = b"""aW1wb3J0IG9zCmltcG9ydCBzeXMNCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBEaWN0LCBBbnks
IE9wdGlvbmFsDQpmcm9tIGZhc3RhcGkgaW1wb3J0IEZhc3RBUEksIEhUVFBFeGNlcHRpb24NCmZy
b20gZmFzdGFwaS5taWRkbGV3YXJlLmNvcnMgaW1wb3J0IENPUlNNaWRkbGV3YXJlDQpmcm9tIHB5
ZGFudGljIGltcG9ydCBCYXNlTW9kZWwsIEZpZWxkDQppbXBvcnQgam9ibGliDQppbXBvcnQgdG9y
Y2gNCmltcG9ydCBudW1weSBhcyBucA0KZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Ub2tl
bml6ZXIsIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24NCg0Kc3lzLnBhdGguYXBw
ZW5kKG9zLnBhdGguYWJzcGF0aChvcy5wYXRoLmpvaW4ob3MucGF0aC5kZXJuYW1lKF9fZmlsZV9f
KSwgIi4uIikpKQ0KZnJvbSBzcmMuZmVhdHVyZXMgaW1wb3J0IGV4dHJhY3RfZmVhdHVyZXNfc2lu
Z2xlLCBGRUFUVVJFX05BTUVTDQoNCmFwcCA9IEZhc3RBUEkodGl0bGU9IlBoaXNoR3VhcmQgQUkg
SW5mZXJlbmNlIEFQSSIsIHZlcnNpb249IjEuMC4wIikNCg0KYXBwLmFkZF9taWRkbGV3YXJlKA0K
ICAgIENPUlNNaWRkbGV3YXJlLA0KICAgIGFsbG93X29yaWdpbnM9WyIqIl0sDQogICAgYWxsb3df
Y3JlZGVudGlhbHM9VHJ1ZSwNCiAgICBhbGxvd19tZXRob2RzPVsiKiJdLA0KICAgIGFsbG93X2hl
YWRlcnM9WyIqIl0sDQopDQoNCk1PREVMT19ESVIgPSBvcy5nZXRlbnYoIk1PREVMT19ESVIiLCAi
L2thZ2dsZS93b3JraW5nL21vZGVscyIpDQpERUNJU0lPTl9USFJFU0hPTEQgPSBmbG9hdChvcy5n
ZXRlbnYoIkRFQ0lTSU9OX1RIUkVTSE9MRCIsICIwLjA3MCIpKQ0KDQpzdGF0ZSA9IHsidGZpZGYi
OiBOb25lLCAieGdiIjogTm9uZSwgIm1ldGFfbGVhcm5lciI6IE5vbmUsICJ0b2tlbml6ZXIiOiBO
b25lLCAiZGlzdGlsX21vZGVsIjogTm9uZSwgImRldmljZSI6ICJjcHVRIn0NCg0KZGVmIGxvYWRf
YXJ0aWZhY3RzKCk6DQogICAgcHJpbnQoIkxvYWRpbmcgUGhpc2hHdWFyZCBJbmZlcmVuY2UgTW9k
ZWxzLi4uIikNCiAgICB0cnk6DQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKGYie01PREVMT19E
SVJ9L2Jhc2VsaW5lX3RmaWRmX2xyLmpvYmxpYiIpOg0KICAgICAgICAgICAgc3RhdGVbInRmaWRm
Il0gPSBqb2JsaWIubG9hZChmInVNPREVMT19ESVJ9L2Jhc2VsaW5lX3RmaWRmX2xyLmpvYmxpYiIp
DQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKGYie01PREVMT19ESVJ9L3hnYl9mZWF0dXJlcy5q
b2JsaWIiKToNCiAgICAgICAgICAgIHN0YXRlWyJ4Z2IiXSA9IGpvYmxpYi5sb2FkKGYie01PREVM
T19ESVJ9L3hnYl9mZWF0dXJlcy5qb2JsaWIiKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhm
IntNT0RFTE9fRElSfS9tZXRhX2xlYXJuZXIuam9ibGliIik6DQogICAgICAgICAgICBzdGF0ZVsi
bWV0YV9sZWFybmVyIl0gPSBqb2JsaWIubG9hZChmIntNT0RFTE9fRElSfS9tZXRhX2xlYXJuZXIu
am9ibGliIikNCiAgICAgICAgZGlzdGlsX3BhdGggPSBmIntNT0RFTE9fRElSfS9kaXN0aWxiZXJ0
X2Jlc3QiDQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKGRpc3RpbF9wYXRoKToNCiAgICAgICAg
ICAgIHN0YXRlWyJ0b2tlbml6ZXIiXSA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGRp
c3RpbF9wYXRoKQ0KICAgICAgICAgICAgc3RhdGVbImRpc3RpbF9tb2RlbCJdID0gQXV0b01vZGVs
Rm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbi5mcm9tX3ByZXRyYWluZWQoZGlzdGlsX3BhdGgpDQog
ICAgICAgICAgICBzdGF0ZVsiZGlzdGlsX21vZGVsIl0uZXZhbCgpDQogICAgICAgICAgICBkZXZp
Y2UgPSAiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiDQogICAg
ICAgICAgICBzdGF0ZVsiZGlzdGlsX21vZGVsIl0udG8oZGV2aWNlKQ0KICAgICAgICAgICAgc3Rh
dGVbImRldmljZSJdID0gZGV2aWNlDQogICAgICAgIHByaW50KCJNb2RlbHMgc3VjY2Vzc2Z1bGx5
IGxvYWRlZCExKQ0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgcHJpbnQoZlJJ
c3N1ZSBsb2FkaW5nIG1vZGVsczogZ2UpDQoNCkFwcC5vbl9ldmVudCgic3RhcnR1cCIpDQpkZWYg
b25fc3RhcnR1cCgpOg0KICAgIGxvYWRfYXJ0aWZhY3RzKCkNCg0KY2xhc3MgUHJlZGljdFJlcXVl
c3QoQmFzZU1vZGVsKToNCiAgICB0ZXh0OiBzdHIgPSBGaWVsZCguLi4sIG1pbl9sZW5ndGg9Mywg
bWF4X2xlbmd0aD0xNTAwMCkNCg0KY2xhc3MgU2lnbmFsRGV0YWlsKEJhc2VNb2RlbCk6DQogICAg
ZmVhdHVyZTogc3RyDQogICAgdmFsdWU6IGZsb2F0DQogICAgZGVzY3JpcHRpb246IHN0cg0KDQpj
bGFzcyBQcmVkaWN0UmVzcG9uc2UoQmFzZU1vZGVsKToNCiAgICB2ZXJkaWN0OiBzdHINCiAgICBy
aXNrX3Njb3JlOiBmbG9hdA0KICAgIHRocmVzaG9sZDogZmxvYXQNCiAgICBpc19waGlzaGluZzog
Ym9vbA0KICAgIGRpc3RpbGJlcnRfcHJvYjogZmxvYXQNCiAgICB4Z2JfcHJvYjogZmxvYXQNCiAg
ICB0ZmlkZl9wcm9iOiBmbG9hdA0KICAgIHRvcF9zaWduYWxzOiBMaXN0W1NpZ25hbERldGFpbF0N
Cg0KQGFwcC5nZXQoIi9oZWFsdGgiKQ0KZGVmIGhlYWx0aCgpOg0KICAgIHJldHVybiB7DQogICAg
ICAgICJzdGF0dXMiOiAiaGVhbHRoeSIsDQogICAgICAgICJkZXZpY2UiOiBzdGF0ZVsiZGV2aWNl
Il0sDQogICAgICAgICJtb2RlbHNfbG9hZGVkIjogew0KICAgICAgICAgICAgInRmaWRmIjogc3Rh
dGVbInRmaWRmIl0gaXMgbm90IE5vbmUsDQogICAgICAgICAgICAieGdiIjogc3RhdGVbInhnYiJd
IGlzIG5vdCBOb25lLA0KICAgICAgICAgICAgIm1ldGFfbGVhcm5lciI6IHN0YXRlWyJtZXRhX2xl
YXJuZXIiXSBpcyBub3QgTm9uZSwNCiAgICAgICAgICAgICJkaXN0aWxiZXJ0Ijogc3RhdGVbImRp
c3RpbF9tb2RlbCJdIGlzIG5vdCBOb25lDQogICAgICAgIH0sDQogICAgICAgICJ0aHJlc2hvbGQi
OiBERUNJU0lPTl9USFJFU0hPTEQNCiAgICB9DQoNCkBhcHAucG9zdCgiL3ByZWRpY3QiLCByZXNw
b25zZV9tb2RlbD1QcmVkaWN0UmVzcG9uc2UpDQpkZWYgcHJlZGljdChwYXlsb2FkOiBQcmVkaWN0
UmVxdWVzdCk6DQogICAgdGV4dCA9IHBheWxvYWQudGV4dC5zdHJpcCgpDQogICAgaWYgbm90IHRl
eHQ6DQogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDAwLCBkZXRhaWw9
IlRleHQgY2Fubm90IGJlIGVtcHR5LiIpDQoNCiAgICBmZWF0X2RpY3QgPSBleHRyYWN0X2ZlYXR1
cmVzX3NpbmdsZSh0ZXh0KQ0KICAgIGZlYXRfdmVjID0gbnAuYXJyYXkoW1tmZWF0X2RpY3Rba10g
Zm9yIGsgaW4gRkVBVFVSRV9OQU1FU11dLCBkdHlwZT1ucC5mbG9hdDMyKQ0KDQogICAgeGdiX3Ag
PSBmbG9hdChzdGF0ZVsieGdiIl0ucHJlZGljdF9wcm9iYShmZWF0X3ZlYylbMCwgMV0pIGlmIHN0
YXRlWyJ4Z2IiXSBlbHNlIDAuNQ0KICAgIHRmaWRmX3AgPSBmbG9hdChzdGF0ZVsidGZpZGYiXS5w
cmVkaWN0X3Byb2JhKFt0ZXh0XSlbMCwgMV0pIGlmIHN0YXRlWyJ0ZmlkZiJdIGVsc2UgMC41DQoN
CiAgICBkaXN0aWxfcCA9IDAuNQ0KICAgIGlmIHN0YXRlWyJkaXN0aWxfbW9kZWwiXSBhbmQgc3Rh
dGVbInRva2VuaXplciJdOg0KICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAg
ICAgIGVuY29kZWQgPSBzdGF0ZVsidG9rZW5pemVyIl0odGV4dCwgdHJ1bmNhdGlvbj1UcnVlLCBt
YXhfbGVuZ3RoPTI1NiwgcGFkZGluZz1UcnVlLCByZXR1cm5fdGVuc29ycz0icHQiKS50byhzdGF0
ZVsiZGV2aWNlIl0pDQogICAgICAgICAgICBsb2dpdHMgPSBzdGF0ZVsiZGlzdGlsX21vZGVsIl0o
KiplbmNvZGVkKS5sb2dpdHMNCiAgICAgICAgICAgIGRpc3RpbF9wID0gZmxvYXQodG9yY2guc29m
dG1heChsb2dpdHMsIGRpbT0tMSlbMCwgMV0uY3B1KCkuaXRlbSgpKQ0KDQogICAgaWYgc3RhdGVb
Im1ldGFfbGVhcm5lciJdOg0KICAgICAgICBtZXRhX2ZlYXR1cmVzID0gbnAuYXJyYXkoW1tkaXN0
aWxfcCwgeGdiX3AsIHRmaWRmX3BdXSkNCiAgICAgICAgZW5zZW1ibGVfc2NvcmUgPSBmbG9hdChz
dGF0ZVsibWV0YV9sZWFybmVyIl0ucHJlZGljdF9wcm9iYShtZXRhX2ZlYXR1cmVzKVswLCAxXSkN
CiAgICBlbHNlOg0KICAgICAgICBlbnNlbWJsZV9zY29yZSA9IGZsb2F0KChkaXN0aWxfcCAqIDAu
NikgKyAoeGdiX3AgKiAwLjQpKQ0KDQogICAgaXNfcGhpc2ggPSBib29sKGVuc2VtYmxlX3Njb3Jl
ID49IERFQ0lTSU9OX1RIUkVTSE9MRCkNCiAgICB2ZXJkaWN0ID0gIlBISVNISU5HIC8gTUFMSUNJ
T1VTIiBpZiBpc19waGlzaCBlbHNlICJMRUdJVElNQVRFIC8gU0FGRSINCg0KICAgIHRvcF9zaWdu
YWxzID0gW10NCiAgICBpZiBmZWF0X2RpY3RbIm51bV91cmxzIl0gPiAwOg0KICAgICAgICB0b3Bf
c2lnbmFscy5hcHBlbmQoU2lnbmFsRGV0YWlsKGZlYXR1cmU9Im51bV91cmxzIiwgdmFsdWU9ZmVh
dF9kaWN0WyJudW1fdXJscyJdLCBkZXNjcmlwdGlvbj1mIkNvbnRhaW5zIHtpbnQoZmVhdF9kaWN0
WydudW1fdXJscyddKX0gaHlwZXJsaW5rKHMpIikpDQogICAgaWYgZmVhdF9kaWN0WyJoYXNfaXBf
dXJsIl0gPT0gMToNCiAgICAgICAgdG9wX3NpZ25hbHMuYXBwZW5kKFNpZ25hbERldGFpbChmZWF0
dXJlPSJoYXNfaXBfdXJsIiwgdmFsdWU9MS4wLCBkZXNjcmlwdGlvbj0iRGlyZWN0IElQIGFkZHJl
c3MgaW4gbGluayBkZXN0aW5hdGlvbiAoSGlnaCBSaXNrKSIpKQ0KICAgIGlmIGZlYXRfZGljdFsia
GFzX3Nob3J0ZW5lciJdID09IDE6DQogICAgICAgIHRvcF9zaWduYWxzLmFwcGVuZChTaWduYWxEZX
RhaWwoZmVhdHVyZT0iaGFzX3Nob3J0ZW5lciIsIHZhbHVlPTEuMCwgZGVzY3JpcHRpb249IlVzZX
MgVVJMcmVkaXJlY3RvciBvciBzaG9ydGVuZXIgc2VydmljZSIpKQ0KICAgIGlmIGZlYXRfZGljdFs
idXJnZW5jeV9oaXRzIl0gPiAwOg0KICAgICAgICB0b3Bfc2lnbmFscy5hcHBlbmQoU2lnbmFsRGV0
YWlsKGZlYXR1cmU9InVyZ2VuY3lfaGl0cyIsIHZhbHVlPWZlYXRfZGljdFsidXJnZW5jeV9oaXRzI
l0sIGRlc2NyaXB0aW9uPWZ7aW50KGZlYXRfZGljdFsndXJnZW5jeV9oaXRzJ10pfSBzb2NpYWwg
ZW5naW5lZXJpbmcgdXJnZW5jeSB0cmlnZ2VyKHMpIikpDQogICAgaWYgZmVhdF9kaWN0WyJ1cHBl
cl9yYXRpbyJdID4gMC4xNToNCiAgICAgICAgdG9wX3NpZ25hbHMuYXBwZW5kKFNpZ25hbERldGFp
bChmZWF0dXJlPSJ1cHBlcl9yYXRpbyIsIHZhbHVlPXJvdW5kKGZlYXRfZGljdFsidXBwZXJfcmF0
aW8iXSwgMiksIGRlc2NyaXB0aW9uPSJBYm5vcm1hbGx5IGhpZ2ggdXBwZXJjYXNlIHJhdGlvIikp
DQoNCiAgICByZXR1cm4gUHJlZGljdFJlc3BvbnNlKA0KICAgICAgICB2ZXJkaWN0PXZlcmRpY3Qs
DQogICAgICAgIHJpc2tfc2NvcmU9cm91bmQoZW5zZW1ibGVfc2NvcmUsIDQpLA0KICAgICAgICB0
aHJlc2hvbGQ9REVDSVNJT05fVEhSRVNIT0xELA0KICAgICAgICBpc19waGlzaGluZz1pc19waGlz
aCwNCiAgICAgICAgZGlzdGlsYmVydF9wcm9iPXJvdW5kKGRpc3RpbF9wLCA0KSwNCiAgICAgICAg
eGdiX3Byb2I9cm91bmQoeGdiX3AsIDQpLA0KICAgICAgICB0ZmlkZl9wcm9iPXJvdW5kKHRmaWRm
X3AsIDQpLA0KICAgICAgICB0b3Bfc2lnbmFscz10b3Bfc2lnbmFscw0KICAgICkNCg=="""

os.makedirs("/kaggle/working/api", exist_ok=True)
with open("/kaggle/working/api/main.py", "wb") as f:
    f.write(base64.b64decode(b64_content))

print("Binary decoded /kaggle/working/api/main.py cleanly written!")

# Run tests
!pytest -v /kaggle/working/tests/test_api.py

Binary decoded /kaggle/working/api/main.py cleanly written!
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
/usr/local/lib/python3.13/dist-packages/_pytest/python.py:498: in importtestmodule
    mod = import_path(
/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:587: in import_path
    importlib.import_module(module_name)
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<frozen impo

In [20]:
# Cell 14: Bulletproof Clean Write
with open("/kaggle/working/api/main.py", "w") as f:
    f.write('''import os
import sys
from typing import List, Dict, Any
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import joblib
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from src.features import extract_features_single, FEATURE_NAMES

app = FastAPI(title="PhishGuard AI Inference API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

MODELS_DIR = os.getenv("MODELS_DIR", "/kaggle/working/models")
DECISION_THRESHOLD = float(os.getenv("DECISION_THRESHOLD", "0.070"))

state = {
    "tfidf": None,
    "xgb": None,
    "meta_learner": None,
    "tokenizer": None,
    "distil_model": None,
    "device": "cpu"
}

def load_artifacts():
    print("Loading models into memory...")
    try:
        tfidf_path = os.path.join(MODELS_DIR, "baseline_tfidf_lr.joblib")
        if os.path.exists(tfidf_path):
            state["tfidf"] = joblib.load(tfidf_path)

        xgb_path = os.path.join(MODELS_DIR, "xgb_features.joblib")
        if os.path.exists(xgb_path):
            state["xgb"] = joblib.load(xgb_path)

        meta_path = os.path.join(MODELS_DIR, "meta_learner.joblib")
        if os.path.exists(meta_path):
            state["meta_learner"] = joblib.load(meta_path)

        distil_path = os.path.join(MODELS_DIR, "distilbert_best")
        if os.path.exists(distil_path):
            state["tokenizer"] = AutoTokenizer.from_pretrained(distil_path)
            state["distil_model"] = AutoModelForSequenceClassification.from_pretrained(distil_path)
            state["distil_model"].eval()
            device = "cuda" if torch.cuda.is_available() else "cpu"
            state["distil_model"].to(device)
            state["device"] = device
        print("Models successfully loaded!")
    except Exception as e:
        print("Model loading notice:", e)

@app.on_event("startup")
def on_startup():
    load_artifacts()

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=3, max_length=15000)

class SignalDetail(BaseModel):
    feature: str
    value: float
    description: str

class PredictResponse(BaseModel):
    verdict: str
    risk_score: float
    threshold: float
    is_phishing: bool
    distilbert_prob: float
    xgb_prob: float
    tfidf_prob: float
    top_signals: List[SignalDetail]

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "device": state["device"],
        "models_loaded": {
            "tfidf": state["tfidf"] is not None,
            "xgb": state["xgb"] is not None,
            "meta_learner": state["meta_learner"] is not None,
            "distilbert": state["distil_model"] is not None
        },
        "threshold": DECISION_THRESHOLD
    }

@app.post("/predict", response_model=PredictResponse)
def predict(payload: PredictRequest):
    text = payload.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Text cannot be empty.")

    feat_dict = extract_features_single(text)
    feat_vec = np.array([[feat_dict[k] for k in FEATURE_NAMES]], dtype=np.float32)

    xgb_p = float(state["xgb"].predict_proba(feat_vec)[0, 1]) if state["xgb"] else 0.5
    tfidf_p = float(state["tfidf"].predict_proba([text])[0, 1]) if state["tfidf"] else 0.5

    distil_p = 0.5
    if state["distil_model"] and state["tokenizer"]:
        with torch.no_grad():
            encoded = state["tokenizer"](<
                text,
                truncation=True,
                max_length=256,
                padding=True,
                return_tensors="pt"
            >).to(state["device"])
            logits = state["distil_model"](**encoded).logits
            distil_p = float(torch.softmax(logits, dim=-1)[0, 1].cpu().item())

    if state["meta_learner"]:
        meta_features = np.array([[distil_p, xgb_p, tfidf_p]])
        ensemble_score = float(state["meta_learner"].predict_proba(meta_features)[0, 1])
    else:
        ensemble_score = float((distil_p * 0.6) + (xgb_p * 0.4))

    is_phish = bool(ensemble_score >= DECISION_THRESHOLD)
    verdict = "PHISHING / MALICIOUS" if is_phish else "LEGITIMATE / SAFE"

    top_signals = []
    if feat_dict["num_urls"] > 0:
        top_signals.append(SignalDetail(feature="num_urls", value=feat_dict["num_urls"], description="Contains " + str(int(feat_dict["num_urls"])) + " hyperlink(s)"))
    if feat_dict["has_ip_url"] == 1:
        top_signals.append(SignalDetail(feature="has_ip_url", value=1.0, description="Direct IP address in link destination (High Risk)"))
    if feat_dict["has_shortener"] == 1:
        top_signals.append(SignalDetail(feature="has_shortener", value=1.0, description="Uses URL redirector or shortener service"))
    if feat_dict["urgency_hits"] > 0:
        top_signals.append(SignalDetail(feature="urgency_hits", value=feat_dict["urgency_hits"], description=str(int(feat_dict["urgency_hits"])) + " social engineering urgency trigger(s)"))
    if feat_dict["upper_ratio"] > 0.15:
        top_signals.append(SignalDetail(feature="upper_ratio", value=round(feat_dict["upper_ratio"], 2), description="Abnormally high uppercase ratio"))

    return PredictResponse(
        verdict=verdict,
        risk_score=round(ensemble_score, 4),
        threshold=DECISION_THRESHOLD,
        is_phishing=is_phish,
        distilbert_prob=round(distil_p, 4),
        xgb_prob=round(xgb_p, 4),
        tfidf_prob=round(tfidf_p, 4),
        top_signals=top_signals
    )
''')

# Write clean test suite
with open("/kaggle/working/tests/test_api.py", "w") as f:
    f.write('''import os
import sys
from fastapi.testclient import TestClient

sys.path.append("/kaggle/working")
from api.main import app, load_artifacts

load_artifacts()
client = TestClient(app)

def test_health_endpoint():
    response = client.get("/health")
    assert response.status_code == 200
    data = response.json()
    assert data["status"] == "healthy"
    assert "models_loaded" in data

def test_predict_phishing():
    payload = {
        "text": "URGENT SECURITY ALERT: Your PayPal account has been suspended. Click http://192.168.1.1/verify to enter your password immediately!"
    }
    response = client.post("/predict", json=payload)
    assert response.status_code == 200
    res = response.json()
    assert res["is_phishing"] is True
    assert res["verdict"] == "PHISHING / MALICIOUS"
    assert len(res["top_signals"]) > 0

def test_predict_safe():
    payload = {
        "text": "Hi team, please find the quarterly report attached. Looking forward to our discussion in tomorrow standup."
    }
    response = client.post("/predict", json=payload)
    assert response.status_code == 200
    res = response.json()
    assert res["risk_score"] < 0.5
''')

print("Files written successfully! Running pytest...")
!pytest -v /kaggle/working/tests/test_api.py

Files written successfully! Running pytest...
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
/usr/local/lib/python3.13/dist-packages/_pytest/python.py:498: in importtestmodule
    mod = import_path(
/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:587: in import_path
    importlib.import_module(module_name)
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<frozen importlib._bootstr

In [21]:
# Cell 14: Use __call__ method to avoid browser paste substitution
api_code = '''import os
import sys
from typing import List, Dict, Any
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import joblib
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from src.features import extract_features_single, FEATURE_NAMES

app = FastAPI(title="PhishGuard AI Inference API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

MODELS_DIR = os.getenv("MODELS_DIR", "/kaggle/working/models")
DECISION_THRESHOLD = float(os.getenv("DECISION_THRESHOLD", "0.070"))

state = {
    "tfidf": None,
    "xgb": None,
    "meta_learner": None,
    "tokenizer": None,
    "distil_model": None,
    "device": "cpu"
}

def load_artifacts():
    print("Loading models into memory...")
    try:
        tfidf_path = os.path.join(MODELS_DIR, "baseline_tfidf_lr.joblib")
        if os.path.exists(tfidf_path):
            state["tfidf"] = joblib.load(tfidf_path)

        xgb_path = os.path.join(MODELS_DIR, "xgb_features.joblib")
        if os.path.exists(xgb_path):
            state["xgb"] = joblib.load(xgb_path)

        meta_path = os.path.join(MODELS_DIR, "meta_learner.joblib")
        if os.path.exists(meta_path):
            state["meta_learner"] = joblib.load(meta_path)

        distil_path = os.path.join(MODELS_DIR, "distilbert_best")
        if os.path.exists(distil_path):
            state["tokenizer"] = AutoTokenizer.from_pretrained(distil_path)
            state["distil_model"] = AutoModelForSequenceClassification.from_pretrained(distil_path)
            state["distil_model"].eval()
            dev = "cuda" if torch.cuda.is_available() else "cpu"
            state["distil_model"].to(dev)
            state["device"] = dev
        print("Models successfully loaded!")
    except Exception as e:
        print("Model loading notice:", e)

@app.on_event("startup")
def on_startup():
    load_artifacts()

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=3, max_length=15000)

class SignalDetail(BaseModel):
    feature: str
    value: float
    description: str

class PredictResponse(BaseModel):
    verdict: str
    risk_score: float
    threshold: float
    is_phishing: bool
    distilbert_prob: float
    xgb_prob: float
    tfidf_prob: float
    top_signals: List[SignalDetail]

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "device": state["device"],
        "models_loaded": {
            "tfidf": state["tfidf"] is not None,
            "xgb": state["xgb"] is not None,
            "meta_learner": state["meta_learner"] is not None,
            "distilbert": state["distil_model"] is not None
        },
        "threshold": DECISION_THRESHOLD
    }

@app.post("/predict", response_model=PredictResponse)
def predict(payload: PredictRequest):
    text = payload.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="Text cannot be empty.")

    feat_dict = extract_features_single(text)
    feat_vec = np.array([[feat_dict[k] for k in FEATURE_NAMES]], dtype=np.float32)

    xgb_p = float(state["xgb"].predict_proba(feat_vec)[0, 1]) if state["xgb"] else 0.5
    tfidf_p = float(state["tfidf"].predict_proba([text])[0, 1]) if state["tfidf"] else 0.5

    distil_p = 0.5
    tok = state["tokenizer"]
    model = state["distil_model"]
    if tok and model:
        with torch.no_grad():
            batch = tok.__call__(text, truncation=True, max_length=256, padding=True, return_tensors="pt")
            batch = {k: v.to(state["device"]) for k, v in batch.items()}
            logits = model.__call__(**batch).logits
            distil_p = float(torch.softmax(logits, dim=-1)[0, 1].cpu().item())

    if state["meta_learner"]:
        meta_features = np.array([[distil_p, xgb_p, tfidf_p]])
        ensemble_score = float(state["meta_learner"].predict_proba(meta_features)[0, 1])
    else:
        ensemble_score = float((distil_p * 0.6) + (xgb_p * 0.4))

    is_phish = bool(ensemble_score >= DECISION_THRESHOLD)
    verdict = "PHISHING / MALICIOUS" if is_phish else "LEGITIMATE / SAFE"

    top_signals = []
    if feat_dict["num_urls"] > 0:
        top_signals.append(SignalDetail(feature="num_urls", value=feat_dict["num_urls"], description="Contains " + str(int(feat_dict["num_urls"])) + " hyperlink(s)"))
    if feat_dict["has_ip_url"] == 1:
        top_signals.append(SignalDetail(feature="has_ip_url", value=1.0, description="Direct IP address in link destination (High Risk)"))
    if feat_dict["has_shortener"] == 1:
        top_signals.append(SignalDetail(feature="has_shortener", value=1.0, description="Uses URL redirector or shortener service"))
    if feat_dict["urgency_hits"] > 0:
        top_signals.append(SignalDetail(feature="urgency_hits", value=feat_dict["urgency_hits"], description=str(int(feat_dict["urgency_hits"])) + " social engineering urgency trigger(s)"))
    if feat_dict["upper_ratio"] > 0.15:
        top_signals.append(SignalDetail(feature="upper_ratio", value=round(feat_dict["upper_ratio"], 2), description="Abnormally high uppercase ratio"))

    return PredictResponse(
        verdict=verdict,
        risk_score=round(ensemble_score, 4),
        threshold=DECISION_THRESHOLD,
        is_phishing=is_phish,
        distilbert_prob=round(distil_p, 4),
        xgb_prob=round(xgb_p, 4),
        tfidf_prob=round(tfidf_p, 4),
        top_signals=top_signals
    )
'''

with open("/kaggle/working/api/main.py", "w") as f:
    f.write(api_code)

print("Saved api/main.py using tok.__call__!")

!pytest -v /kaggle/working/tests/test_api.py

Saved api/main.py using tok.__call__!
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 3 items                                                              

tests/test_api.py::test_health_endpoint PASSED                           [ 33%]
tests/test_api.py::test_predict_phishing PASSED                          [ 66%]
tests/test_api.py::test_predict_safe PASSED                              [100%]

=============================== warnings summary ===============================
api/main.py:64
  /kaggle/working/api/main.py:64: DeprecationWarning: 
          on_event is deprecated, use lifespan event handlers instead.
  
          Read more about it in the
          [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
          
    @ap

In [22]:
# Create a deployment archive containing the lightweight trained models
import shutil
import os

deploy_models = "/kaggle/working/deploy_models"
os.makedirs(deploy_models, exist_ok=True)

# Copy the trained weights
for f in ["baseline_tfidf_lr.joblib", "xgb_features.joblib", "meta_learner.joblib"]:
    src_f = f"/kaggle/working/models/{f}"
    if os.path.exists(src_f):
        shutil.copy(src_f, deploy_models)

# Create a zip to easily download
shutil.make_archive("/kaggle/working/phishguard_models", "zip", deploy_models)
print("Models zip ready at /kaggle/working/phishguard_models.zip")

Models zip ready at /kaggle/working/phishguard_models.zip
